In [1]:
import openai
from openai import OpenAI
import os
import json
import random
import glob

In [ ]:
with open("_secret_key4", "r") as f:
    openai_key = f.read()
os.environ["OPENAI_API_KEY"]=openai_key

In [3]:


# Define the directories
nber_dir = "../../../2_raw_data/1_nber/nber_pdf_text_all_processed_filt/"
cepr_dir = "../../../2_raw_data/2_cepr/cepr_pdf_text_all_processed_filt/"


# get all the data in 1 place

In [4]:
import pandas as pd 
# Step 1: Read all files into DataFrames
df = pd.read_csv('../Data/Speeches/all_cb_speeches.csv',sep='\t') 

# iterate through each line to obtain the sentences

In [5]:

import tqdm
import re 

import re

def split_sentences(text):
    # Common exceptions where a period does not end a sentence
    exceptions = [
        "Mr", "Mrs", "Ms", "Dr", "Prof", "Sr", "Jr", "St", "Mt", "vs",
        "etc", "e.g", "i.e", "Fig", "Inc", "Ltd", "Co", "No", "U.S", "U.K"
    ]

    # Pattern: a period followed by space and uppercase letter
    # We'll split here only if the part before the period is not in the exceptions
    pattern = re.compile(r'(?<!\w\.\w)(?<![A-Z][a-z]\.)(?<=\.)\s+(?=[A-Z])')

    # First, split using the safe pattern
    candidates = pattern.split(text)

    # Then fix any false splits by merging pieces that were wrongly split
    repaired = []
    for i, part in enumerate(candidates):
        if i == 0:
            repaired.append(part)
        else:
            prev = repaired[-1].strip()
            last_word = prev.split()[-1].rstrip('.')  # Get the last word before the split
            if last_word in exceptions or re.match(r'^[A-Z]\.?$', last_word):  # e.g., 'Dr', 'J.'
                # Merge back
                repaired[-1] = prev + ' ' + part
            else:
                repaired.append(part)

    return [s.strip() for s in repaired if s.strip()]


# Corrected regex to split sentences
#sentences = split_sentences(df['text'][0])#re.split(r'(?<!\b(?:Dr|Mr|Mrs|Ms|St))(?<!\d)\. (?=[A-Z])', all_data['text'][0])
#for i in df.index:
#    sentences = split_sentences(df['text'][i])
#    df.at[i, 'sentences'] = sentences

In [6]:
#Add a new column with the split sentences.
def add_split_sentences(df, text_col='text'):
    df['split_sentences'] = df[text_col].apply(split_sentences)
    return df

# Batch Function: Create Sentence Batches
def create_batches_from_sentences(sentences, batch_len=None):
    if batch_len==None:
        batch_len = max(int(len(sentences)*0.1),1)
        return [sentences[i:i + batch_len] for i in range(0, len(sentences), batch_len)]
    else :
        return [sentences[i:i + batch_len] for i in range(0, len(sentences), batch_len)]
        

# Master Function: Return Dictionary Per Row

def prepare_batches(df, batch_len, text_col='text'):
    df = add_split_sentences(df, text_col)
    output = []

    for _, row in tqdm.tqdm(df.iterrows(),total=len(df), desc="Processing Rows"):
        batches = create_batches_from_sentences(row['split_sentences'], batch_len)
        output.append({
            'speech_id': row['speech_id'],
            'batches': batches
        })

    return output


In [ ]:
df


,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source,date,speech_id
0,https://www.bankofgreece.gr/en/news-and-media/...,NaN,"Opening remarks at the ""GBA Session on Non-Per...",NaN,2019-09-11,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Opening remarks by Dimitris Malliaropulos, Chi...",NaN,grc_dimitris_malliaropulos_11_09_2019_english,English,CB websites,2019-09-11,1
1,https://www.bankofgreece.gr/en/news-and-media/...,NaN,"Opening Remarks at the IFFR Conference: ""Under...",NaN,2019-09-12,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Opening Remarks by Dimitris Malliaropulos, Chi...",NaN,grc_dimitris_malliaropulos_12_09_2019_english,English,CB websites,2019-09-12,2
2,https://www.bankofgreece.gr/enimerosi/grafeio-...,NaN,"Speech at the GetInvolved conference: ""The dec...",NaN,2019-12-13,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Speech by Dimitris Malliaropoulos, Chief Econo...",", & GetInvolved : ...",grc_dimitris_malliaropulos_13_12_2019_greek,Greek,CB websites,2019-12-13,3
3,https://www.bankofgreece.gr/en/news-and-media/...,NaN,Speech at the AHK Europa Konferenz: Remarks on...,NaN,2019-09-20,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Speech by Dimitris Malliaropulos, Chief Econom...",NaN,grc_dimitris_malliaropulos_20_09_2019_english,English,CB websites,2019-09-20,4
4,https://www.bankofgreece.gr/en/news-and-media/...,NaN,"Speech: ""Assessing the performance and regulat...",NaN,2010-02-04,Eleni D Dendrinou-Louri,Deputy Governor,Female,Bank of Greece,GRC,"Speech of the Dep. Governor E. Louri: ""Assessi...",NaN,grc_eleni_d_dendrinou-louri_04_02_2010_english,English,CB websites,2010-02-04,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35482,NaN,https://www.boz.zm/PCF-dissemination.pdf,Speech at 2018 FPI & Investor Perceptions diss...,NaN,2018-12-05,Denny H Kalyalya,Governor,Male,Bank of Zambia,ZMB,2018 DISSEMINATION WORKSHOP ON FOREIGN PRIVATE...,NaN,zmb_pcf-dissemination,English,CB websites,2018-12-05,35483
35483,NaN,https://www.boz.zm/Speech_on_the_launch_of_the...,Official launch of the National Financial Switch,NaN,2019-06-19,Bwalya K E Ng'andu,Deputy Governor,Male,Bank of Zambia,ZMB,OFFICIAL LAUNCHING OF GOING LIVE OF THE NATION...,NaN,zmb_speech_on_the_launch_of_the_nfs_final,English,CB websites,2019-06-19,35484
35484,NaN,https://www.boz.zm/Speech.pdf,Speech on Credit Reporting Education and Aware...,NaN,2018-10-19,Denny H Kalyalya,Governor,Male,Bank of Zambia,ZMB,LAUNCH OF THE CREDIT REPORTING EDUCATION AND A...,NaN,zmb_speechoctober2018zambia,English,CB websites,2018-10-19,35485
35485,NaN,https://www.boz.zm/Talking_Notes_for_Deputy_Go...,Speech: Launch of the 2018 Digital Financial S...,NaN,2019-04-11,Bwalya K E Ng'andu,Deputy Governor,Male,Bank of Zambia,ZMB,"DR BWALYA NGANDU DEPUTY GOVERNOR - OPERATIONS,...",NaN,zmb_talking_notes_for_deputy_governor_launch_o...,English,CB websites,2019-04-11,35486


In [8]:
# Prepare batches
results = prepare_batches(df, batch_len=None, text_col='text')

# Example output
for item in results:
    print(item)
    break

Processing Rows: 100%|█████████████████| 35487/35487 [00:03<00:00, 10311.45it/s]

{'speech_id': 1, 'batches': [['Opening remarks by Dimitris Malliaropulos, Chief Economist and Director of economic analysis and research at the "GBA Session on Non-Performing Loans", organized by the London Business School at the Bank of Greece. 11/09/2019 -Speeches It is a great pleasure to welcome you today to the 2ndGBA session on Non-Performing Loans organized by the London Business School here at the Bank of Greece.', 'Unlike last year, where the focus of the GBA session was on the banks role and the dynamics of the financial sector, the focus this year will be more on the role of NPLs as a driver of the Greek economy dynamics and as a source of opportunity moving forward.', 'The steep rise in NPLs in Greece over the past decade was largely the result of the economic crisis, particularly its depth and duration.', 'Between 2008 and 2016, Greece lost over 25% of real GDP and the unemployment rate rose by nearly 16 percentage points.', 'The deterioration in the macro-economy caused s

In [9]:
results_df = pd.DataFrame(results).explode('batches')

In [10]:
len(df['split_sentences'].iloc[0])

51

In [11]:
len(item['batches'])

11

In [12]:
item['batches'][0]

['Opening remarks by Dimitris Malliaropulos, Chief Economist and Director of economic analysis and research at the "GBA Session on Non-Performing Loans", organized by the London Business School at the Bank of Greece. 11/09/2019 -Speeches It is a great pleasure to welcome you today to the 2ndGBA session on Non-Performing Loans organized by the London Business School here at the Bank of Greece.',
 'Unlike last year, where the focus of the GBA session was on the banks role and the dynamics of the financial sector, the focus this year will be more on the role of NPLs as a driver of the Greek economy dynamics and as a source of opportunity moving forward.',
 'The steep rise in NPLs in Greece over the past decade was largely the result of the economic crisis, particularly its depth and duration.',
 'Between 2008 and 2016, Greece lost over 25% of real GDP and the unemployment rate rose by nearly 16 percentage points.',
 'The deterioration in the macro-economy caused serious debt-servicing pro

In [13]:
item['batches'][-2]

['A substantial reduction of NPLs is a precondition for banks to improve their rating scores, thus allowing access to private funding at a lower cost which can be passed to companies and households in form of lower interest rates for bank credit.',
 'Let me close with two last remarks regarding loan restructurings and reforms of the regulatory framework of NPLs: Although sales, primarily through securitizations and loan write-offs are the main drivers of the NPL reduction so far, this should not lead banks to relax their efforts to successfully restructure the loans that are still in their portfolios.',
 'On this front, progress is rather slow and banks need to come up with improved long-term loan modification schemes so that the observed high rate of re-defaults is contained.',
 'Finally, more work is needed to further improve the regulatory framework so as to remove bureaucratic obstacles and legal loop holes.',
 'The new regime of primary residence protection must fit into a single 

In [14]:
causal_mechanism_message = {
    "role": "system",
    "content": (
        "You are an expert assistant specializing in identifying causal mechanisms in economics reports. "
        "For each received text, extract EVERY causal mechanism discussed (explicit or implicit) and return them "
        "as a list under a top-level field named 'claims'. Each element in 'claims' is one causal mechanism "
            "extracted from the input text, covering macroeconomic, fiscal, monetary, structural, institutional, or financial-sector domains.\n\n"
        "**Definition of a Causal Mechanism:**\n"
        "Any explicit or implicit statement describing how one economic variable, event, or policy action directly or indirectly influences another variable or outcome. This causal statement should **NOT** be descriptive, a statment that says that variable [X] has increased or will increase is not causal under this definition. However a statement that describes that variable [X] will increase because of variable [Y] is a causal statement.\n\n"
        "**Important Instructions:**\n"
        "- Be exhaustive: include macroeconomic, fiscal, monetary, structural, institutional, and financial-sector mechanisms.\n"
        "- For EACH mechanism, populate these fields:\n"
        "  - **cause_entity**: The initiating economic variable, event, or policy decision.\n"
        "  - **effect_entity**: The resulting impact or change on another variable, sector, or indicator.\n"
        "  - **relationship**: A precise description of the causal relationship linking cause_entity to effect_entity.\n"
        "  - **document_reference**: Cite the reference sentence(s) the claim is taken from.\n"
        "  - **quantifications**: Any numerical evidence, statistics, or measurable outcomes supporting the claim (if present).\n"
        "  - **modulator_uncertainty**: The optional statement where the speaker expresses their tentativeness or uncertainty about the causal claim.\n"
         "  - **modulator_temporal**: The optional statement where the speaker expresses their belief in the time span of the causal claim.\n"
        "  - **tense**: Whether the speaker believes the causal relationship has happened (past), is happening (present) or will happen (futur).\n"
        "- Return a SINGLE top-level JSON object with a field 'claims' that is a LIST of mechanism objects (one object per mechanism).\n"
        "- If no mechanisms are present, return {\"claims\": []}.\n"
        "- Use standalone, self-contained phrasing; do not invent content beyond the text.\n"
        "- Fill all fields; if absent in the text, write 'NA'.\n\n"

        "**Output Format (STRICT):**\n"
        "{\n"
        "  \"claims\": [\n"
        "    {\n"
        "      \"cause_entity\": \"...\",\n"
        "      \"effect_entity\": \"...\",\n"
        "      \"relationship\": \"...\",\n"
        "      \"document_reference\": \"...\",\n"
        "      \"quantifications\": \"...\",\n"
        "      \"modulator_uncertainty\":\"...\",\n"
        "      \"modulator_temporal\":\"...\",\n"
        "      \"tense\":\"...\",\n"
        "    }\n"
        "  ]\n"
        "}\n"
    )
}


response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "causal_mechanism_extraction_speeches_v2",
        "strict": True,
        "description": (
            "Return a single top-level object containing a 'claims' array. Each element in 'claims' is one causal mechanism "
            "extracted from the input text, covering macroeconomic, fiscal, monetary, structural, institutional, or financial-sector domains. An causal statement of the type [X] and [Y] cause [Z] should be written out as two separate causal claims [X] causes [Z] and [Y] causes [Z]. Similarly, [X] causes [A] and [B] should be written as separate claims [X] causes [A] and [X] causes [B]."
        ),
        "schema": {
            "type": "object",
            "properties": {
                "claims": {
                    "type": "array",
                    "description": "List of all causal mechanisms found in the provided text.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "cause_entity": {
                                "type": "string",
                                "description": (
                                    "The initiating economic entity, variable, event, or policy decision that sets the mechanism in motion. Reword pronouns to what the pronouns are referring to."
                                )
                            },
                            "effect_entity": {
                                "type": "string",
                                "description": (
                                    "The affected economic entity, variable sector or indicator resulting from the causal relationship."
                                )
                            },
                            "relationship": {
                                "type": "string",
                                "description": (
                                    "Concise, precise economic logic or transmission channel explaining how the causal_entity causaly affects the effect_entity."
                                )
                            },
                            "document_reference": {
                                "type": "string",
                                "description": (
                                    "The sentence which the relationship is drawn from."
                                )
                            },
                            "quantifications": {
                                "type": "string",
                                "description": (
                                    "Specific numerical evidence/statistics supporting the causal claim (percentages, coefficients, magnitudes). "
                                    "Use 'NA' if none provided."
                                )
                            },
                            "modulator_uncertainty": {
                                "type": "string",
                                #"enum": ["Low", "Medium", "High"],
                                "description": (
                                    "An optional modulator to the causal statement made by the speaker, composed of conditionals statements like 'if', 'perhaps' etc. It could also be a hedging statment using modals like 'can', 'could', 'might' etc or paraphrases like 'it is possible' etc."
                                    "Use 'NA' if none provided."
                                )
                            },
                            "modulator_temporal": {
                                "type": "string",
                                #"enum": ["Low", "Medium", "High"],
                                "description": (
                                    "An optional modulator to the causal statement made by the speaker about the duration of the causal relationship, this refers to statements by the speaker suggesting that the causal relationship may be temporary, quote the duration the speaker describes."
                                    "Use 'NA' if none provided."
                                )
                            },
                        "tense": {
                                "type": "string",
                                "enum": ["PAST", "PRESENT", "FUTURE"],
                                "description": (
                                    "An optional qualifier to the causal claim describing whether the causal relationship happenned in the past [PAST], is happenning in the present [PRESENT] or will take place in the future [FUTURE]."
                                    "Use 'NA' if none provided."
                                )
                            }
                        },
                        "required": [
                            "cause_entity",
                            "effect_entity",
                            "relationship",
                            "document_reference",
                            "quantifications",
                            "modulator_uncertainty",
                            "modulator_temporal",
                            "tense"
                        ],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["claims"],
            "additionalProperties": False
        }
    }
}


In [ ]:
# ==== Centralise model + generation params (TOP OF CELL) ====
import os
import json
from uuid import uuid4
from datetime import datetime
from zoneinfo import ZoneInfo
import csv

import openai
# If you need: from openai import OpenAI

# --- Choose your model here ---
MODEL = "gpt-4o"#os.getenv("OPENAI_MODEL", "gpt-5")  # e.g., "gpt-4o" or "gpt-5o"
MAX_TOKENS = 8000

# Derive param overrides from the chosen model
def _gen_params_for_model(model_name: str, max_tokens: int):
    """
    Returns (temperature, extra_body_fields_dict).
    - For gpt-4o: temperature=0,   body uses {"max_tokens": max_tokens}
    - For gpt-5o: temperature=1,   body uses {"max_completion_tokens": max_tokens}
    - Fallback:   temperature=0.7, body uses {"max_tokens": max_tokens}
    """
    name = (model_name or "").lower().replace(" ", "")
    if "gpt-4o" in name or name == "gpt4o":
        return 0, {"max_tokens": max_tokens}
    if "gpt-5" in name or name == "gpt5":
        return 1, {"max_completion_tokens": max_tokens}
    # sensible fallback
    return 0.7, {"max_tokens": max_tokens}

TEMPERATURE, MODEL_TOKEN_FIELD = _gen_params_for_model(MODEL, MAX_TOKENS)

# --- API client (set up once) ---

client = openai.OpenAI() 

# ---- Optional conveniences / constants ----
LONDON_TZ = ZoneInfo("Europe/London")
BATCH_LOG_CSV = "batch_submissions.csv"

# You should have these defined somewhere in your notebook:
# causal_mechanism_message = {"role": "system", "content": "..."}
# response_format_schema = {...}  # if you’re using JSON schema mode
# response_format = {"type": "json_object"}  # if you’re using generic JSON mode


# ===================== Helpers =====================
def _extract_requests_from_jsonl(jsonl_file_path):
    with open(jsonl_file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            custom_id = obj.get("custom_id")
            try:
                user_msg = obj["body"]["messages"][-1]
                text_content = user_msg.get("content", "")
            except Exception:
                text_content = ""
            yield custom_id, text_content

def _append_batch_log(csv_path, batch_id, jsonl_file_path, sent_at=None):
    sent_at = sent_at or datetime.now(LONDON_TZ).isoformat()
    new_file = not os.path.exists(csv_path)

    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(["batch_id", "custom_id", "sent_at_iso", "text"])
        for custom_id, text_content in _extract_requests_from_jsonl(jsonl_file_path):
            writer.writerow([batch_id, custom_id, sent_at, text_content])


# ===================== Core: JSONL generation =====================
def generate_batch_jsonl(
    speech_id,
    batches_data,
    prompt,
    output_path="openai_batch.jsonl",
    model: str = MODEL,
    use_json_schema=True,
    fallback_to_json_object=True,
    response_format_schema=None,
    max_tokens: int = MAX_TOKENS,
):
    """
    Build a JSONL file for the Batch API.
    Uses the centralized MODEL/TEMPERATURE and applies the correct token field
    depending on whether the model is gpt-4o or gpt-5.
    """
    # Recompute in case caller overrides model/max_tokens
    temperature, token_field = _gen_params_for_model(model, max_tokens)

    speech_batches = next((item for item in batches_data if item['speech_id'] == speech_id), None)
    if not speech_batches:
        raise ValueError(f"No batches found for speech_id {speech_id}")

    with open(output_path, "w", encoding="utf-8") as f:
        for idx, batch in enumerate(speech_batches['batches']):
            batch_text = " ".join(batch)

            full_user_content = (
                "Here is the text extract of a speech made by a central banker:\n\n"
                f"Batch {idx+1}:\n{batch_text}\n\n"
                f"Task:\n{prompt.strip()}"
            )

            # Base body
            body = {
                "model": model,
                "messages": [
                    causal_mechanism_message,  # ensure defined in your notebook
                    {"role": "user", "content": full_user_content}
                ],
                "temperature": temperature,
                # token field injected below (either "max_tokens" or "max_completion_tokens")
            }
            body.update(token_field)

            # Response format handling
            if use_json_schema and response_format_schema is not None:
                body["response_format"] = response_format_schema
            elif fallback_to_json_object:
                body["response_format"] = {"type": "json_object"}

            request_payload = {
                "custom_id": f"{speech_id}_batch_{idx+1}_{uuid4()}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": body,
            }
            f.write(json.dumps(request_payload) + "\n")

    print(f"Batch file written to {output_path}")
    return output_path


# ===================== Batch submit / status / download =====================
def submit_batch_job(jsonl_file_path, metadata=None):
    """
    Submit the batch job using openai>=1.0.0 syntax,
    then log (batch_id, text, time) to CSV.
    """
    input_file = client.files.create(
        file=open(jsonl_file_path, "rb"),
        purpose="batch"
    )

    batch = client.batches.create(
        input_file_id=input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata=metadata or {"type": "speech_batch_job"}
    )

    print(f"Batch submitted with ID: {batch.id}")

    _append_batch_log(BATCH_LOG_CSV, batch_id=batch.id, jsonl_file_path=jsonl_file_path)
    return batch


def get_batch_status(batch_id):
    batch = client.batches.retrieve(batch_id)
    print(f"Status: {batch.status}")
    return batch


def download_batch_output(batch, save_path=None):
    """
    Download the completed batch output.
    NOTE: must use batch.output_file_id, not batch.id
    """
    if getattr(batch, "output_file_id", None) is None:
        raise ValueError(f"Batch {batch.id} has no output_file_id yet (status={batch.status}).")

    output_file_id = batch.output_file_id
    _ = client.files.retrieve(file_id=output_file_id)
    content_stream = client.files.content(output_file_id)

    save_path = save_path or f"output_{batch.id}.jsonl"
    with open(save_path, "wb") as f:
        f.write(content_stream.read())
    print(f"Output saved to {save_path}")
    return save_path





In [55]:
import os
import csv
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
import json

# Reuse your timezone constant if already defined
LONDON_TZ = ZoneInfo("Europe/London")


# ---------- CSV helpers ----------
def _ensure_status_csv(csv_path: str):
    exists = os.path.exists(csv_path)
    if not exists:
        with open(csv_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "batch_id",
                "status",
                "ready_to_retrieve",
                "completed",
                "completed_at_time",
                "tokens_submitted",
                "tokens_output",
                "output_path",
            ])

def _read_status_rows(csv_path: str):
    if not os.path.exists(csv_path):
        return []
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

def _write_status_rows(csv_path: str, rows: list[dict]):
    fieldnames = [
        "batch_id",
        "status",
        "ready_to_retrieve",
        "completed",
        "completed_at_time",
        "tokens_submitted",
        "tokens_output",
        "output_path",
    ]
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fieldnames})

def _upsert_status_row(csv_path: str, batch_id: str, status: str, ready_to_retrieve: bool):
    rows = _read_status_rows(csv_path)
    idx = next((i for i, r in enumerate(rows) if r["batch_id"] == batch_id), None)
    base = {
        "batch_id": batch_id,
        "status": status,
        "ready_to_retrieve": "YES" if ready_to_retrieve else "NO",
        "completed": rows[idx]["completed"] if idx is not None else "NO",
        "completed_at_time": rows[idx]["completed_at_time"] if idx is not None else "",
        "tokens_submitted": rows[idx]["tokens_submitted"] if idx is not None else "",
        "tokens_output": rows[idx]["tokens_output"] if idx is not None else "",
        "output_path": rows[idx]["output_path"] if idx is not None else "",
    }
    if idx is None:
        rows.append(base)
    else:
        rows[idx].update(base)
    _write_status_rows(csv_path, rows)

# ---------- Output parsing ----------
def _sum_tokens_from_output_jsonl(jsonl_path: str):
    """
    Sum prompt/completion tokens from a Batch output JSONL.
    Handles typical shapes:
      line -> { "response": { "body": { "usage": {...} } } } or
      line -> { "response": { "body": { "data": [...]} } } (rare)
    Returns (prompt_total, completion_total).
    """
    prompt_total = 0
    completion_total = 0
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            usage = None
            # Common case
            try:
                usage = obj["response"]["body"].get("usage")
            except Exception:
                usage = None
            # Sometimes responses are nested per item; try to aggregate
            if usage is None:
                try:
                    items = obj["response"]["body"].get("data", [])
                    for item in items:
                        u = item.get("usage")
                        if u:
                            prompt_total += int(u.get("prompt_tokens", 0))
                            completion_total += int(u.get("completion_tokens", 0))
                    continue
                except Exception:
                    pass
            if usage:
                prompt_total += int(usage.get("prompt_tokens", 0))
                completion_total += int(usage.get("completion_tokens", 0))
    return prompt_total, completion_total

# ---------- Completed processor ----------
def _process_completed_rows(csv_path: str, download_dir: str):
    """
    For any row whose batch is completed and not yet marked 'completed' == YES,
    download the output, sum tokens, and update the row.
    """
    rows = _read_status_rows(csv_path)
    changed = False
    Path(download_dir).mkdir(parents=True, exist_ok=True)

    for r in rows:
        # Skip already marked completed
        if r.get("completed", "NO") == "YES":
            continue

        batch_id = r["batch_id"]
        # Double-check status live (in case CSV is stale)
        try:
            batch = client.batches.retrieve(batch_id)
            status = getattr(batch, "status", "unknown")
            r["status"] = status
        except Exception as e:
            # Keep previous status; continue
            continue

        # Only proceed if completed and has output file id
        output_file_id = getattr(batch, "output_file_id", None)
        if r["status"] == "completed" and output_file_id:
            # Download using your helper, to user-specified folder
            save_path = os.path.join(download_dir, f"output_{batch.id}.jsonl")
            try:
                downloaded_path = download_batch_output(batch, save_path=save_path)
            except Exception as e:
                # If download fails, skip this row for now
                continue

            # Parse tokens from the downloaded file
            prompt_toks, completion_toks = _sum_tokens_from_output_jsonl(downloaded_path)

            # Update row
            r["completed"] = "YES"
            r["completed_at_time"] = datetime.now(LONDON_TZ).isoformat()
            r["tokens_submitted"] = str(prompt_toks)
            r["tokens_output"] = str(completion_toks)
            r["output_path"] = downloaded_path
            changed = True

    if changed:
        _write_status_rows(csv_path, rows)

# ---------- Main monitor ----------
def monitor_batches(
    batch_ids: list[str],
    download_dir: str,
    status_csv: str = "batch_runs.csv",
    check_every: int = 100,
):
    """
    - Iterates through batch_ids, retrieves status, updates CSV.
    - Prints progress updates.
    - Every `check_every` IDs (default 100), scans CSV for completed rows,
      downloads outputs, computes token totals, marks completed.
    - Runs a final scan at the end as well.
    """
    _ensure_status_csv(status_csv)
    total = len(batch_ids)
    for i, bid in enumerate(batch_ids, start=1):
        try:
            batch = client.batches.retrieve(bid)
            status = getattr(batch, "status", "unknown")
            ready = getattr(batch, "output_file_id", None) is not None
            _upsert_status_row(status_csv, bid, status, ready)
            print(f"[{i}/{total}] {bid}: status={status}, ready_to_retrieve={'YES' if ready else 'NO'}")
        except Exception as e:
            # Record as unknown if retrieval failed
            _upsert_status_row(status_csv, bid, "error", False)
            print(f"[{i}/{total}] {bid}: error retrieving status ({e})")

        if i % check_every == 0:
            print(f"--- Checkpoint at {i} IDs: processing completed rows ---")
            _process_completed_rows(status_csv, download_dir)

    # Final sweep
    print("--- Final sweep: processing completed rows ---")
    _process_completed_rows(status_csv, download_dir)
    print(f"Done. Status CSV: {status_csv}")


In [56]:
def run_and_log_batches(
    results,                    # list[{"speech_id": ..., "batches": [...]}]
    prompt: str,
    download_dir: str,          # where outputs should be saved when completed
    status_csv: str = "batch_runs.csv",
    model: str | None = None,
    max_tokens: int | None = None,
    include_only_speech_ids: list[int] | None = None,
    check_every: int = 100,
    speech_index_csv: str = "speech_index.csv",
    error_csv: str = "error_batches.csv",
):
    """
    Behavior:
      - If a speech was submitted before (found in `speech_index.csv`):
          * If batch 'completed':
              - ensure output file is present (download if needed)
              - validate output (no per-line errors, status_code==200)
              - if valid => SKIP re-run
              - if invalid or batch failed => append to error_batches.csv
          * If not completed => SKIP resubmission; we'll monitor it
      - If speech is new => generate, submit, log, add to monitor list

    Uses your existing helpers: _ensure_status_csv, _upsert_status_row, monitor_batches,
    download_batch_output, and the global `client` (OpenAI).
    """
    import csv, os
    from pathlib import Path
    from datetime import datetime

    # ---------- local helpers ----------
    def _ensure_csv_with_header(path: str, header: list[str]):
        if not os.path.exists(path):
            with open(path, "w", encoding="utf-8", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(header)

    def _read_csv_dicts(path: str) -> list[dict]:
        if not os.path.exists(path):
            return []
        with open(path, "r", encoding="utf-8", newline="") as f:
            return list(csv.DictReader(f))

    def _write_csv_dicts(path: str, rows: list[dict], header: list[str]):
        with open(path, "w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=header)
            w.writeheader()
            for r in rows:
                w.writerow({k: r.get(k, "") for k in header})

    def _index_get(index_rows: list[dict], speech_id: int | str):
        sid = str(speech_id)
        for r in index_rows:
            if r.get("speech_id") == sid:
                return r
        return None

    def _index_upsert(index_rows: list[dict], speech_id: int | str, batch_id: str, last_action: str):
        sid = str(speech_id)
        now_iso = datetime.now(LONDON_TZ).isoformat()
        header = ["speech_id", "batch_id", "last_checked_at", "last_action"]
        row = _index_get(index_rows, sid)
        if row is None:
            index_rows.append({
                "speech_id": sid,
                "batch_id": batch_id,
                "last_checked_at": now_iso,
                "last_action": last_action,
            })
        else:
            row["batch_id"] = batch_id
            row["last_checked_at"] = now_iso
            row["last_action"] = last_action
        return header

    def _record_error(error_rows: list[dict], speech_id: int | str, batch_id: str, reason: str, output_path: str = ""):
        header = ["detected_at_iso", "speech_id", "batch_id", "reason", "output_path"]
        error_rows.append({
            "detected_at_iso": datetime.now(LONDON_TZ).isoformat(),
            "speech_id": str(speech_id),
            "batch_id": batch_id,
            "reason": reason,
            "output_path": output_path,
        })
        return header

    def _find_output_path_in_status(status_rows: list[dict], batch_id: str) -> str | None:
        for r in status_rows:
            if r.get("batch_id") == batch_id:
                p = r.get("output_path")
                if p:
                    return p
        return None

    def _batch_output_has_errors(jsonl_path: str):
        """
        Returns (has_errors: bool, error_count: int, first_reason: str|None)
        We flag an error if:
          - top-level has 'error', or
          - response.status_code != 200, or
          - response.body has an 'error' field.
        """
        import json
        count = 0
        first = None
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    count += 1
                    if first is None:
                        first = "invalid JSON line"
                    continue

                # 1) top-level error
                if "error" in obj and obj["error"]:
                    count += 1
                    if first is None:
                        first = "top-level error in batch line"
                    continue

                resp = obj.get("response", {})
                scode = resp.get("status_code")
                if scode is not None and scode != 200:
                    count += 1
                    if first is None:
                        first = f"status_code={scode}"
                    continue

                body = resp.get("body", {})
                if isinstance(body, dict) and "error" in body and body["error"]:
                    count += 1
                    if first is None:
                        first = "error in response body"
                    continue

        return (count > 0), count, first

    # ---------- setup ----------
    Path(download_dir).mkdir(parents=True, exist_ok=True)
    _ensure_status_csv(status_csv)

    # index + error ledgers
    _ensure_csv_with_header(speech_index_csv, ["speech_id", "batch_id", "last_checked_at", "last_action"])
    _ensure_csv_with_header(error_csv, ["detected_at_iso", "speech_id", "batch_id", "reason", "output_path"])

    index_rows = _read_csv_dicts(speech_index_csv)
    error_rows = _read_csv_dicts(error_csv)
    status_rows = _read_status_rows(status_csv)

    # ---------- main loop ----------
    to_monitor_batch_ids = []
    submitted_count = 0

    for item in results:
        sid = item.get("speech_id")
        if include_only_speech_ids and sid not in include_only_speech_ids:
            continue

        prior = _index_get(index_rows, sid)

        if prior is not None:
            # Previously submitted: check status first
            prev_batch_id = prior["batch_id"]
            try:
                batch = client.batches.retrieve(prev_batch_id)
                status = getattr(batch, "status", "unknown")
                ready = getattr(batch, "output_file_id", None) is not None
                _upsert_status_row(status_csv, prev_batch_id, status, ready)
            except Exception as e:
                # couldn't retrieve the old batch; mark error and move on (do not resubmit)
                _record_error(error_rows, sid, prev_batch_id, f"retrieve_failed: {e}")
                _index_upsert(index_rows, sid, prev_batch_id, "retrieve_failed")
                print(f"[skip:retrieve_failed] speech_id={sid} batch_id={prev_batch_id}")
                continue

            if status == "completed":
                # Ensure we have the output file path
                out_path = _find_output_path_in_status(status_rows, prev_batch_id)
                if not out_path or not os.path.exists(out_path):
                    # download now
                    save_path = os.path.join(download_dir, f"output_{batch.id}.jsonl")
                    try:
                        out_path = download_batch_output(batch, save_path=save_path)
                        # Update tokens + completed fields in status CSV
                        prompt_toks, completion_toks = _sum_tokens_from_output_jsonl(out_path)
                        # Update all rows in status CSV for this batch
                        for r in status_rows:
                            if r.get("batch_id") == prev_batch_id:
                                r["completed"] = "YES"
                                r["completed_at_time"] = datetime.now(LONDON_TZ).isoformat()
                                r["tokens_submitted"] = str(prompt_toks)
                                r["tokens_output"] = str(completion_toks)
                                r["output_path"] = out_path
                        _write_status_rows(status_csv, status_rows)
                    except Exception as e:
                        _record_error(error_rows, sid, prev_batch_id, f"download_failed: {e}")
                        _index_upsert(index_rows, sid, prev_batch_id, "download_failed")
                        print(f"[skip:download_failed] speech_id={sid} batch_id={prev_batch_id}")
                        continue

                # Validate contents
                has_err, err_cnt, reason = _batch_output_has_errors(out_path)
                if has_err:
                    _record_error(error_rows, sid, prev_batch_id, f"completed_with_errors ({err_cnt}): {reason}", out_path)
                    _index_upsert(index_rows, sid, prev_batch_id, "completed_with_errors")
                    print(f"[skip:error_output] speech_id={sid} batch_id={prev_batch_id} reason={reason}")
                    continue
                else:
                    # Completed and valid -> skip resubmission
                    _index_upsert(index_rows, sid, prev_batch_id, "completed_valid_skip")
                    print(f"[skip:completed_valid] speech_id={sid} batch_id={prev_batch_id}")
                    continue

            elif status in {"failed", "expired", "canceled", "cancelled", "error"}:
                _record_error(error_rows, sid, prev_batch_id, f"status={status}")
                _index_upsert(index_rows, sid, prev_batch_id, f"status_{status}")
                print(f"[skip:prior_failed] speech_id={sid} batch_id={prev_batch_id} status={status}")
                continue

            else:
                # queued/running/in_progress -> do not resubmit; monitor
                to_monitor_batch_ids.append(prev_batch_id)
                _index_upsert(index_rows, sid, prev_batch_id, f"in_progress({status})")
                print(f"[monitor] speech_id={sid} batch_id={prev_batch_id} status={status}")
                continue

        # -------- new submission path --------
        # Build a per-speech JSONL path
        jsonl_path = f"speech_{sid}_batches.jsonl"
        kwargs = {}
        if model is not None:
            kwargs["model"] = model
        if max_tokens is not None:
            kwargs["max_tokens"] = max_tokens

        # Generate JSONL for this speech
        generate_batch_jsonl(
            speech_id=sid,
            batches_data=results,
            prompt=prompt,
            output_path=jsonl_path,
            **kwargs,
        )

        # Submit the batch & capture id
        batch = submit_batch_job(jsonl_path)
        submitted_count += 1

        # Log initial status row
        try:
            live = get_batch_status(batch.id)
            status = getattr(live, "status", "submitted")
            ready = getattr(live, "output_file_id", None) is not None
        except Exception:
            status, ready = ("submitted", False)

        _upsert_status_row(status_csv, batch.id, status, ready)
        to_monitor_batch_ids.append(batch.id)

        # Update index
        _index_upsert(index_rows, sid, batch.id, "submitted")
        print(f"[submitted] speech_id={sid} -> batch_id={batch.id} (status={status})")

    # ---------- persist index and error ledgers ----------
    _write_csv_dicts(speech_index_csv, index_rows, ["speech_id", "batch_id", "last_checked_at", "last_action"])
    _write_csv_dicts(error_csv, error_rows, ["detected_at_iso", "speech_id", "batch_id", "reason", "output_path"])

    # ---------- monitor all pending batches ----------
    if to_monitor_batch_ids:
        monitor_batches(
            batch_ids=to_monitor_batch_ids,
            download_dir=download_dir,
            status_csv=status_csv,
            check_every=check_every,
        )
    else:
        print("Nothing to monitor (all speeches were previously completed/failed).")
    print(f"Submitted new batches: {submitted_count}")


In [ ]:
# Generate JSONL File 
import json
from uuid import uuid4

import openai

# Set your API key (do not hard-code this in production!)

client = openai.OpenAI(api_key=openai_key)

def generate_batch_jsonl(speech_id, batches_data, prompt, output_path="openai_batch.jsonl", model="gpt-5"):
    max_tokens = 200
    speech_batches = next((item for item in batches_data if item['speech_id'] == speech_id), None)
    if not speech_batches:
        raise ValueError(f"No batches found for speech_id {speech_id}")
    
    with open(output_path, "w", encoding="utf-8") as f:
        for idx, batch in enumerate(speech_batches['batches']):
            batch_text = " ".join(batch)
            full_prompt = f"""
            \n\nBatch {idx+1}:\n{batch_text}
            """
            #f"{prompt}\n\nBatch {idx+1}:\n{batch_text}"

            request_payload = {
                "custom_id": f"{speech_id}_batch_{idx+1}_{uuid4()}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model,
                    "messages" : [causal_mechanism_message,
                                {"role": "user",
                                 "content": f"Here is the text extract of a speech made by a central banker:\n\n{full_prompt}"
                                }
                               ],

                    
                    #"messages": [
                    #    {"role": "system", "content": "You are a macroeconomic policy analyst reviewing a speech that was given by a central bank representative or governor report."},
                    #    {"role": "user", "content": full_prompt}
                    #],
                    "temperature": 0,
                    "max_completion_tokens": 8000,
                    "response_format": response_format  # Use the updated response format
                }
            }
            f.write(json.dumps(request_payload) + "\n")
    
    print(f"Batch file written to {output_path}")
    return output_path


def submit_batch_job(jsonl_file_path, metadata=None):
    """
    Submit the batch job using openai>=1.0.0 syntax.
    """
    # Upload the JSONL input file
    input_file = client.files.create(
        file=open(jsonl_file_path, "rb"),
        purpose="batch"
    )

    # Submit the batch job
    batch = client.batches.create(
        input_file_id=input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata=metadata or {"type": "speech_batch_job"}
    )

    print(f"Batch submitted with ID: {batch.id}")
    return batch

def get_batch_status(batch_id):
    batch = client.batches.retrieve(batch_id)
    print(f"Status: {batch.status}")
    return batch

def download_batch_output(batch):
    output_file_id = batch.id
    output_file = client.files.retrieve(file_id=output_file_id)
    content = client.files.content(output_file.id)
    
    save_path = f"output_{batch.id}.jsonl"
    with open(save_path, "wb") as f:
        f.write(content.read())
    print(f"Output saved to {save_path}")


# Generate JSONL File 
import json
from uuid import uuid4

import openai
# from openai import OpenAI  # if you prefer explicit import style


def generate_batch_jsonl(
    speech_id,
    batches_data,
    prompt,
    output_path="openai_batch.jsonl",
    model="gpt-5",
    use_json_schema=True,
    fallback_to_json_object=True,
    response_format_schema=None,
    max_tokens=2000,
):
    """
    Build a JSONL file for the Batch API. If the selected model does not support
    response_format=json_schema, set `use_json_schema=False` or keep True and allow
    fallback by setting `fallback_to_json_object=True`.
    """
    speech_batches = next((item for item in batches_data if item['speech_id'] == speech_id), None)
    if not speech_batches:
        raise ValueError(f"No batches found for speech_id {speech_id}")

    with open(output_path, "w", encoding="utf-8") as f:
        for idx, batch in enumerate(speech_batches['batches']):
            batch_text = " ".join(batch)

            # keep your separate task prompt + batch payload explicit
            full_user_content = (
                "Here is the text extract of a speech made by a central banker:\n\n"
                f"Batch {idx+1}:\n{batch_text}\n\n"
                f"Task:\n{prompt.strip()}"
            )

            body = {
                "model": model,
                "messages": [
                    # expects you defined `causal_mechanism_message` earlier; if not, replace with your system message
                    causal_mechanism_message,
                    {"role": "user", "content": full_user_content}
                ],
                "temperature": 0,
                "max_completion_tokens": max_tokens,
            }

            # Attach response_format according to capability/flags
            if use_json_schema and response_format_schema is not None:
                # Try json_schema
                body["response_format"] = response_format_schema
            elif fallback_to_json_object:
                # Force JSON mode without schema enforcement
                body["response_format"] = {"type": "json_object"}

            request_payload = {
                "custom_id": f"{speech_id}_batch_{idx+1}_{uuid4()}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": body,
            }
            f.write(json.dumps(request_payload) + "\n")

    print(f"Batch file written to {output_path}")
    return output_path


def submit_batch_job(jsonl_file_path, metadata=None):
    """
    Submit the batch job using openai>=1.0.0 syntax.
    """
    input_file = client.files.create(
        file=open(jsonl_file_path, "rb"),
        purpose="batch"
    )

    batch = client.batches.create(
        input_file_id=input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata=metadata or {"type": "speech_batch_job"}
    )

    print(f"Batch submitted with ID: {batch.id}")
    return batch


def get_batch_status(batch_id):
    batch = client.batches.retrieve(batch_id)
    print(f"Status: {batch.status}")
    return batch


def download_batch_output(batch, save_path=None):
    """
    Download the completed batch output.
    NOTE: must use batch.output_file_id, not batch.id
    """
    if getattr(batch, "output_file_id", None) is None:
        raise ValueError(f"Batch {batch.id} has no output_file_id yet (status={batch.status}).")

    output_file_id = batch.output_file_id
    # Retrieve and download content
    _ = client.files.retrieve(file_id=output_file_id)
    content_stream = client.files.content(output_file_id)

    save_path = save_path or f"output_{batch.id}.jsonl"
    with open(save_path, "wb") as f:
        f.write(content_stream.read())
    print(f"Output saved to {save_path}")
    return save_path


# Generate JSONL File 
import json
from uuid import uuid4
import os
import csv
from datetime import datetime
from zoneinfo import ZoneInfo

import openai
# Assumes you've created `client` elsewhere with your custom httpx/cacert/kerberos setup

LONDON_TZ = ZoneInfo("Europe/London")
BATCH_LOG_CSV = "batch_submissions.csv"

def _extract_requests_from_jsonl(jsonl_file_path):
    """
    Read the JSONL you’re about to submit and yield (custom_id, text_content) for each line.
    We pull the user message content so you can see exactly what text was run.
    """
    with open(jsonl_file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            custom_id = obj.get("custom_id")
            try:
                # messages = [system, user]; we take user content
                user_msg = obj["body"]["messages"][-1]  # last message should be the user one
                text_content = user_msg.get("content", "")
            except Exception:
                text_content = ""
            yield custom_id, text_content

def _append_batch_log(csv_path, batch_id, jsonl_file_path, sent_at=None):
    """
    Append rows to CSV: one per request line in the JSONL.
    Creates the CSV with header if it doesn't exist.
    """
    sent_at = sent_at or datetime.now(LONDON_TZ).isoformat()
    new_file = not os.path.exists(csv_path)

    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(["batch_id", "custom_id", "sent_at_iso", "text"])
        for custom_id, text_content in _extract_requests_from_jsonl(jsonl_file_path):
            writer.writerow([batch_id, custom_id, sent_at, text_content])

def generate_batch_jsonl(
    speech_id,
    batches_data,
    prompt,
    output_path="openai_batch.jsonl",
    model="gpt-4o",
    use_json_schema=True,
    fallback_to_json_object=True,
    response_format_schema=None,
    max_tokens=2000,
):
    """
    Build a JSONL file for the Batch API. If the selected model does not support
    response_format=json_schema, set `use_json_schema=False` or keep True and allow
    fallback by setting `fallback_to_json_object=True`.
    """
    speech_batches = next((item for item in batches_data if item['speech_id'] == speech_id), None)
    if not speech_batches:
        raise ValueError(f"No batches found for speech_id {speech_id}")

    with open(output_path, "w", encoding="utf-8") as f:
        for idx, batch in enumerate(speech_batches['batches']):
            batch_text = " ".join(batch)

            full_user_content = (
                "Here is the text extract of a speech made by a central banker:\n\n"
                f"Batch {idx+1}:\n{batch_text}\n\n"
                f"Task:\n{prompt.strip()}"
            )

            body = {
                "model": model,
                "messages": [
                    # expects you defined `causal_mechanism_message` earlier; if not, replace with your system message
                    causal_mechanism_message,
                    {"role": "user", "content": full_user_content}
                ],
                "temperature": 0,
                "max_completion_tokens": max_tokens,
            }

            if use_json_schema and response_format_schema is not None:
                body["response_format"] = response_format_schema
            elif fallback_to_json_object:
                body["response_format"] = {"type": "json_object"}

            request_payload = {
                "custom_id": f"{speech_id}_batch_{idx+1}_{uuid4()}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": body,
            }
            f.write(json.dumps(request_payload) + "\n")

    print(f"Batch file written to {output_path}")
    return output_path

def submit_batch_job(jsonl_file_path, metadata=None):
    """
    Submit the batch job using openai>=1.0.0 syntax,
    then log (batch_id, text, time) to CSV.
    """
    input_file = client.files.create(
        file=open(jsonl_file_path, "rb"),
        purpose="batch"
    )

    batch = client.batches.create(
        input_file_id=input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata=metadata or {"type": "speech_batch_job"}
    )

    print(f"Batch submitted with ID: {batch.id}")

    # ---- NEW: write/update CSV log ----
    _append_batch_log(BATCH_LOG_CSV, batch_id=batch.id, jsonl_file_path=jsonl_file_path)

    return batch

def get_batch_status(batch_id):
    batch = client.batches.retrieve(batch_id)
    print(f"Status: {batch.status}")
    return batch

def download_batch_output(batch, save_path=None):
    """
    Download the completed batch output.
    NOTE: must use batch.output_file_id, not batch.id
    """
    if getattr(batch, "output_file_id", None) is None:
        raise ValueError(f"Batch {batch.id} has no output_file_id yet (status={batch.status}).")

    output_file_id = batch.output_file_id
    _ = client.files.retrieve(file_id=output_file_id)
    content_stream = client.files.content(output_file_id)

    save_path = save_path or f"output_{batch.id}.jsonl"
    with open(save_path, "wb") as f:
        f.write(content_stream.read())
    print(f"Output saved to {save_path}")
    return save_path


In [59]:
df['Authorname'].value_counts().head(5)

Authorname
Alan Greenspan         494
Jean-Claude Trichet    360
Amando M Tetangco      330
Gent Sejko             322
Ignazio Visco          312
Name: count, dtype: int64

In [60]:
df[df['Authorname']=='Catherine L Mann']#.URL.values

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source,date,speech_id,split_sentences
16648,https://www.bankofengland.co.uk/speech/2022/ap...,NaN,A monetary policymaker faces uncertainty spee...,NaN,2022-04-21,Catherine L Mann,Board member,Female,Bank of England,GBR,A monetary policymaker faces uncertainty - spe...,NaN,gbr_catherine_l_mann_44672,English,CB websites,2022-04-21,16649,[A monetary policymaker faces uncertainty - sp...
16649,https://www.bankofengland.co.uk/speech/2022/ju...,NaN,UK monetary policy in the context of global sp...,NaN,2022-06-20,Catherine L Mann,Board member,Female,Bank of England,GBR,UK monetary policy in the context of global sp...,NaN,gbr_catherine_l_mann_44732,English,CB websites,2022-06-20,16650,[UK monetary policy in the context of global s...
16650,https://www.bankofengland.co.uk/speech/2022/se...,NaN,"Inflation expectations, inflation persistence,...",NaN,2022-09-05,Catherine L Mann,Board member,Female,Bank of England,GBR,"Inflation expectations, inflation persistence,...",NaN,gbr_catherine_l_mann_44809,English,CB websites,2022-09-05,16651,"[Inflation expectations, inflation persistence..."
16651,https://www.bankofengland.co.uk/speech/2023/fe...,NaN,Turning Points and Monetary Policy Strategy - ...,NaN,2023-02-06,Catherine L Mann,Board member,Female,Bank of England,GBR,Published on 06 February 2023 Speech Introduct...,NaN,gbr_catherine_l_mann_44963,English,CB websites,2023-02-06,16652,[Published on 06 February 2023 Speech Introduc...
16652,https://www.bankofengland.co.uk/speech/2023/fe...,https://www.bankofengland.co.uk/-/media/boe/fi...,"Expectations, lags, and the transmission of mo...",NaN,2023-02-23,Catherine L Mann,Board member,Female,Bank of England,GBR,"Bank of England Page 1 Expectations, lags, and...",NaN,gbr_catherine_l_mann_44980,English,CB websites,2023-02-23,16653,"[Bank of England Page 1 Expectations, lags, an..."
16653,https://www.bankofengland.co.uk/speech/2023/se...,NaN,Inflation models and research: distilling dyna...,NaN,2023-09-11,Catherine L Mann,Board member,Female,Bank of England,GBR,A journey of 1000 miles begins with a single s...,NaN,gbr_catherine_l_mann_45180,English,CB websites,2023-09-11,16654,[A journey of 1000 miles begins with a single ...
16654,https://www.bankofengland.co.uk/speech/2023/no...,NaN,Climate policy and monetary policy: interactio...,NaN,2023-11-13,Catherine L Mann,Board member,Female,Bank of England,GBR,Climate policy and monetary policy: interactio...,NaN,gbr_catherine_l_mann_45243,English,CB websites,2023-11-13,16655,[Climate policy and monetary policy: interacti...


In [61]:
df.loc[16652]

URL                https://www.bankofengland.co.uk/speech/2023/fe...
PDF                https://www.bankofengland.co.uk/-/media/boe/fi...
Title              Expectations, lags, and the transmission of mo...
Subtitle                                                         NaN
Date                                                      2023-02-23
Authorname                                          Catherine L Mann
Role                                                    Board member
Gender                                                        Female
CentralBank                                          Bank of England
Country                                                          GBR
text               Bank of England Page 1 Expectations, lags, and...
text_original                                                    NaN
Filename                                  gbr_catherine_l_mann_44980
Language                                                     English
Source                            

In [62]:
df[df['speech_id']==16653]

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source,date,speech_id,split_sentences
16652,https://www.bankofengland.co.uk/speech/2023/fe...,https://www.bankofengland.co.uk/-/media/boe/fi...,"Expectations, lags, and the transmission of mo...",NaN,2023-02-23,Catherine L Mann,Board member,Female,Bank of England,GBR,"Bank of England Page 1 Expectations, lags, and...",NaN,gbr_catherine_l_mann_44980,English,CB websites,2023-02-23,16653,"[Bank of England Page 1 Expectations, lags, an..."


In [63]:
df[df['speech_id']==16653]

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source,date,speech_id,split_sentences
16652,https://www.bankofengland.co.uk/speech/2023/fe...,https://www.bankofengland.co.uk/-/media/boe/fi...,"Expectations, lags, and the transmission of mo...",NaN,2023-02-23,Catherine L Mann,Board member,Female,Bank of England,GBR,"Bank of England Page 1 Expectations, lags, and...",NaN,gbr_catherine_l_mann_44980,English,CB websites,2023-02-23,16653,"[Bank of England Page 1 Expectations, lags, an..."


In [64]:
for i in df['split_sentences'].loc[16654]:
    print(i)
    print('\n\n\n------\n\n')
    break

Climate policy and monetary policy: interactions and implications - speech by Catherine L. Mann Given at the Environmental Economics Seminar, University of Oxford Published on 13 November 2023 Catherine L. Mann talks about the importance of considering climate change for monetary policy.



------




In [65]:
[i for i in results if i['speech_id']==16653]

[{'speech_id': 16653,
  'batches': [['Bank of England Page 1 Expectations, lags, and the transmission of monetary policy - speech by Catherine L. Mann Given at the Resolution Foundation 23 February 2023 Bank of England Page 2 Speech 1.',
    'Introduction Economists often reference the long and variable lags of monetary policy, first introduced by Milton Friedman in 1961.',
    'In the central banking world, 18 to 24 months is often quoted as how long it takes for changes in monetary policy to feed through to inflation, even as certainly this effect accumulates over that timeframe.',
    'Although this has by now become a sort of folk wisdom, the economic and policy environment over the past few years has prompted me to re-examine these long and variable lags.',
    'As Ive noted in numerous previous speeches1, the speed and magnitude of monetary transmission depends on the underlying structure of the economy, shocks the economy faces, and on the behaviour of financial markets, firms a

In [66]:
results[16653]#['batches']#[0]

{'speech_id': 16654,
 'batches': [['A journey of 1000 miles begins with a single step: filling gaps in the central bank liquidity toolkit - speech by Andrew Hauser Given at a Market News International Connect Event, Chartered Accountants Hall, London Published on 28 September 2023 Andrew Hauser sets out the Banks ambitious plans for tackling systemic risks in market- based finance by developing a new lending tool for non-bank financial institutions, starting with UK insurance companies and pension funds, including newly-resilient LDI funds.',
   'The tool, which will require the support of market participants and regulators, will be designed to address dysfunction in core sterling markets in the exceptional circumstances where there is a threat to UK financial stability.',
   'It will not absolve firms of their responsibility to maintain robust self-insurance - indeed firms ability to access the facility, and the terms they face, will depend on their level of resilience.',
   'Speech I

In [67]:
prompt = "Extract the causal statements made by the speaker according to your instructions."
jsonl_file = generate_batch_jsonl(
    speech_id=16653,
    batches_data=results,
    prompt=prompt,
    output_path="speech_batches.jsonl",
    # optional overrides:
    # model="gpt-5",
    # max_tokens=500,
)
batch = submit_batch_job(jsonl_file)

status = get_batch_status(batch.id)

Batch file written to speech_batches.jsonl
Batch submitted with ID: batch_69542fab87308190a136b1c41482a620
Status: validating


In [359]:
prompt = "Extract the causal statements made by the speaker according to your instructions."

run_and_log_batches(
    results=results,
    prompt=prompt,
    download_dir="user_outputs/speech_runs",
    status_csv="batch_runs.csv",
    #model="gpt-4o",        # optional
    #max_tokens=500,        # optional
    include_only_speech_ids=[16649, 16650, 16651, 16652, 16653, 16654, 16655],  # run just this one
    check_every=100,
)


Batch file written to speech_16649_batches.jsonl
Batch submitted with ID: batch_68c5d2f670a88190bf6cfff6cb186c25
Status: validating
[submitted] speech_id=16649 -> batch_id=batch_68c5d2f670a88190bf6cfff6cb186c25 (status=validating)
Batch file written to speech_16650_batches.jsonl
Batch submitted with ID: batch_68c5d2f8bcdc8190b1fe67f28af94c2b
Status: validating
[submitted] speech_id=16650 -> batch_id=batch_68c5d2f8bcdc8190b1fe67f28af94c2b (status=validating)
Batch file written to speech_16651_batches.jsonl
Batch submitted with ID: batch_68c5d2fa519c8190899762492151b9eb
Status: validating
[submitted] speech_id=16651 -> batch_id=batch_68c5d2fa519c8190899762492151b9eb (status=validating)
[skip:completed_valid] speech_id=16652 batch_id=batch_68c5c2b02e608190999508ed5f723bc6
[skip:completed_valid] speech_id=16653 batch_id=batch_68c5c2d6a35c819083c5be24fc1d6650
Batch file written to speech_16654_batches.jsonl
Batch submitted with ID: batch_68c5d2fc14cc81908eb563a21ab7b242
Status: validating
[

In [96]:
jsonl_file = generate_batch_jsonl(
    speech_id=16653,
    batches_data=results,
    prompt=""
)

batch = submit_batch_job(jsonl_file)


Batch file written to openai_batch.jsonl
Batch submitted with ID: batch_68c56362880c8190881e911909ee5344


In [206]:
jsonl_file

'speech_batches.jsonl'

In [98]:
prompt = """
Your task is to comprehensively identify and extract all causal mechanisms discussed within the report.  
 
Definition of a causal mechanism:
Any explicit or implicit statement in the report describing how one economic variable, event, or policy action directly or indirectly influences another variable or outcome.
 
For each mechanism identified, detail the following clearly:
 
- Cause: The initiating economic variable, event, or policy decision.  
- Effect: The resulting impact or change on another economic variable, sector, or indicator.  
- Linking Explanation: A concise and precise description of the economic logic, transmission channels, or intermediate steps that connect the cause to the effect.  
- Document Reference: Accurate citation of the location within the document (paragraph, section heading, or page number) for traceability.  
- Quantifications: Specific numerical quantifications, statistics, or measurable outcomes explicitly mentioned in the document supporting the causal claims (if provided).
 
Output Format:  
Provide each mechanism as structured JSON objects clearly separating each component. Ensure extraction covers all types of mechanisms discussed, including macroeconomic, fiscal, monetary, structural, institutional, and financial-sector mechanisms.

"""

In [33]:
batch

Batch(id='batch_69542b1277608190be127bdb7574dcfb', completion_window='24h', created_at=1767123730, endpoint='/v1/chat/completions', input_file_id='file-VNrZfxbptcy3uzbUd78s3R', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1767210130, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'type': 'speech_batch_job'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), model=None, usage={'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens_details': {'reasoning_tokens': 0}})

In [71]:
status = get_batch_status(batch.id)

Status: completed


In [72]:
batchnum = download_batch_output(status)

Output saved to output_batch_69542fab87308190a136b1c41482a620.jsonl


In [92]:
import json

def read_batch_output(file_path):
    results = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)

            # Skip if request failed
            if data.get("response", {}).get("status_code") != 200:
                continue

            # Extract the raw assistant message
            body = data["response"]["body"]
            content = body["choices"][0]["message"]["content"]

            # If you used JSON mode / json_schema, parse it
            try:
                parsed = json.loads(content)
            except json.JSONDecodeError:
                print('json decode failed')
                parsed = content  # fallback to raw text if parsing fails

            results.append({
                "custom_id": data["custom_id"],
                "content": content,
                "parsed": parsed
            })

    return results

# Usage
output_file = batchnum #"output_batch_68c58f4531888190b508a6dca0415667.jsonl"
parsed_results = read_batch_output(output_file)

for r in parsed_results:
    print(f"--- {r['custom_id']} ---")
    print(r["parsed"])


json decode failed
--- 16653_batch_1_68352647-0fc2-4720-be15-0b08ffadc542 ---
{
  "claims": [
    {
      "cause_entity": "Changes in monetary policy",
      "effect_entity": "Inflation",
      "relationship": "It takes 18 to 24 months for changes in monetary policy to feed through to inflation.",
      "document_reference": "Economists often reference the long and variable lags of monetary policy, first introduced by Milton Friedman in 1961. In the central banking world, 18 to 24 months is often quoted as how long it takes for changes in monetary policy to feed through to inflation, even as certainly this effect accumulates over that timeframe.",
      "quantifications": "18 to 24 months",
      "modulator_uncertainty": "NA",
      "modulator_temporal": "NA",
      "tense": "futur"
    },
    {
      "cause_entity": "Structure of the economy, shocks the economy faces, and behavior of financial markets, firms, and households",
      "effect_entity": "Speed and magnitude of monetary tra

In [74]:
len(parsed_results)

11

In [91]:
parsed_results

[{'custom_id': '16653_batch_1_68352647-0fc2-4720-be15-0b08ffadc542',
  'content': '{\n  "claims": [\n    {\n      "cause_entity": "Changes in monetary policy",\n      "effect_entity": "Inflation",\n      "relationship": "It takes 18 to 24 months for changes in monetary policy to feed through to inflation.",\n      "document_reference": "Economists often reference the long and variable lags of monetary policy, first introduced by Milton Friedman in 1961. In the central banking world, 18 to 24 months is often quoted as how long it takes for changes in monetary policy to feed through to inflation, even as certainly this effect accumulates over that timeframe.",\n      "quantifications": "18 to 24 months",\n      "modulator_uncertainty": "NA",\n      "modulator_temporal": "NA",\n      "tense": "futur"\n    },\n    {\n      "cause_entity": "Structure of the economy, shocks the economy faces, and behavior of financial markets, firms, and households",\n      "effect_entity": "Speed and magnit

In [90]:
import ast 
parsed_results[0]['content']

'{\n  "claims": [\n    {\n      "cause_entity": "Changes in monetary policy",\n      "effect_entity": "Inflation",\n      "relationship": "It takes 18 to 24 months for changes in monetary policy to feed through to inflation.",\n      "document_reference": "Economists often reference the long and variable lags of monetary policy, first introduced by Milton Friedman in 1961. In the central banking world, 18 to 24 months is often quoted as how long it takes for changes in monetary policy to feed through to inflation, even as certainly this effect accumulates over that timeframe.",\n      "quantifications": "18 to 24 months",\n      "modulator_uncertainty": "NA",\n      "modulator_temporal": "NA",\n      "tense": "futur"\n    },\n    {\n      "cause_entity": "Structure of the economy, shocks the economy faces, and behavior of financial markets, firms, and households",\n      "effect_entity": "Speed and magnitude of monetary transmission",\n      "relationship": "The speed and magnitude of 

In [84]:
dataframes = []
for index_ in range(len(parsed_results)):
    try :
        #display(pd.DataFrame([pd.Series(i) for i in json.loads(parsed_results[index_]['content'])['claims']]))
            #print()
            #pass
            #print(i)
        print(len([i for i in json.loads(parsed_results[index_]['content'])['claims']]))
        dataframes.append(pd.DataFrame([pd.Series(i) for i in json.loads(parsed_results[index_]['content'])['claims']] ))
    except Exception as e: 
        print('exception',e,index_)
        #print("exception",parsed_results[index_])

exception Unterminated string starting at: line 131 column 7 (char 9619) 0
7
5
10
5
7
6
6
10
0
0


In [80]:
pd.concat(dataframes,ignore_index=True)

,cause_entity,effect_entity,relationship,document_reference,quantifications,modulator_uncertainty,modulator_temporal,tense
0,Tightening in financial conditions,Households cut back on consumption,A tightening in financial conditions should me...,A tightening in financial conditions should me...,NA,should,NA,futur
1,Tightening in financial conditions,Firms reduce investment,"Firms may reduce investment, both in response ...","Firms may reduce investment, both in response ...",NA,may,NA,futur
2,Expectations about the future,Influence the present,Expectations about the future can influence th...,An often underappreciated feature of macroecon...,NA,can,NA,present
3,Expectations,Affect wages and prices directly,Expectations can affect wages and prices direc...,Expectations can affect wages and prices direc...,NA,can,NA,present
4,Shocks outside of central bank control,Influence key channels of the transmission mec...,"Shocks, illustrated on the right-hand side of ...","However, there are also shocks, illustrated on...",NA,can,NA,present
5,Changes in Bank Rate,Affect the real economy and inflation,Changes in Bank Rate affect the real economy a...,"In recent years, economies and central banks a...",NA,most probably,NA,present
6,Monetary conditions tightened,Response to MPC's Bank Rate increases,Monetary conditions have tightened significant...,"Over the last year and a half, monetary condit...",NA,NA,NA,past
7,Tightening from a loose monetary stance,Effect on inflation,Tightening from a loose stance has less of an ...,But research also finds that the extent to whi...,NA,NA,NA,present
8,Falling policy rates and QE,Widening of mortgage spreads,Spreads widened as reference rates fell in res...,Spreads widened as reference rates fell in res...,NA,NA,NA,past
9,Changes in policy rates,Rates households face on their mortgages,There is a lagged pass-through from changes in...,"This is evidence, at least over the sample per...",NA,NA,NA,present


In [349]:
import os, re, glob, json
import pandas as pd

MASTER_COLS = [
    "speech_id", "batch_id",
    "cause_entity", "effect_entity", "relationship",
    "document_reference", "quantifications",
    "modulator_uncertainty", "modulator_temporal", "tense",
    "claim_id",
"unique_identifier_speech_batch_claim",
]

UNIQUE_KEYS = [
    "speech_id", "batch_id",
    "cause_entity", "effect_entity", "relationship",
    "document_reference", "tense",
]

# --- helper: reuse your existing reader ---
def read_batch_output(file_path):
    results = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            if data.get("response", {}).get("status_code") != 200:
                continue
            body = data["response"]["body"]
            content = body["choices"][0]["message"]["content"]
            try:
                parsed = json.loads(content)
            except json.JSONDecodeError:
                parsed = content
            results.append({
                "custom_id": data["custom_id"],
                "content": content,
                "parsed": parsed
            })
    return results

# --- helper: parse "speechId_batch_{n}_{uuid}" robustly ---
_custom_id_re = re.compile(r"^(?P<sid>.+?)_batch_(?P<batchnum>\d+)_", re.IGNORECASE)

def _parse_ids_from_custom_id(custom_id: str):
    m = _custom_id_re.match(custom_id or "")
    if not m:
        return None, None
    sid = m.group("sid")
    # Try to coerce numeric speech_id if possible
    try:
        sid_val = int(sid)
    except Exception:
        sid_val = sid
    return sid_val, int(m.group("batchnum"))

def _ensure_master_df(master_csv: str) -> pd.DataFrame:
    if os.path.exists(master_csv):
        df = pd.read_csv(master_csv, dtype=str).fillna("")
        # coerce types for speech_id/batch_id
        if "speech_id" in df.columns:
            df["speech_id"] = pd.to_numeric(df["speech_id"], errors="ignore")
        if "batch_id" in df.columns:
            df["batch_id"]  = pd.to_numeric(df["batch_id"], errors="coerce").fillna(0).astype(int)
        return df
    return pd.DataFrame(columns=MASTER_COLS)

def _claims_from_parsed(parsed):
    """
    parsed is either dict with 'claims' or raw string.
    Returns iterable of claim dicts (possibly empty).
    """
    if isinstance(parsed, dict) and isinstance(parsed.get("claims"), list):
        return parsed["claims"]
    # if content wasn't JSON, nothing to extract
    return []

def process_outputs_folder(
    folder: str,
    master_csv: str = "claims_master.csv",
    pattern: str = "output_*.jsonl",
) -> pd.DataFrame:
    """
    Scan folder for batch outputs, append unseen claims to master CSV (deduped).
    Returns the up-to-date master DataFrame.
    """
    master_df = _ensure_master_df(master_csv)

    new_rows = []
    for path in sorted(glob.glob(os.path.join(folder, pattern))):
        parsed_results = read_batch_output(path)
        for item in parsed_results:
            speech_id, per_speech_batch = _parse_ids_from_custom_id(item.get("custom_id", ""))
            if speech_id is None or per_speech_batch is None:
                # Skip lines with unexpected custom_id
                continue

            id_counter_unique = 0

            for claim in _claims_from_parsed(item.get("parsed")):
                row = {
                    "speech_id": speech_id,
                    "batch_id": per_speech_batch,
                    "claim_id":id_counter_unique,
                    "unique_identifier_speech_batch_claim": f"{speech_id}_{per_speech_batch}_{id_counter_unique}",
                    "cause_entity":             str(claim.get("cause_entity", "")),
                    "effect_entity":            str(claim.get("effect_entity", "")),
                    "relationship":             str(claim.get("relationship", "")),
                    "document_reference":       str(claim.get("document_reference", "")),
                    "quantifications":          str(claim.get("quantifications", "")),
                    "modulator_uncertainty":    str(claim.get("modulator_uncertainty", "")),
                    "modulator_temporal":       str(claim.get("modulator_temporal", "")),
                    "tense":                    str(claim.get("tense", "")),
                }
                new_rows.append(row)
                id_counter_unique+=1

    if new_rows:
        new_df = pd.DataFrame(new_rows, columns=MASTER_COLS)
        # Normalize dtypes
        new_df["speech_id"] = pd.to_numeric(new_df["speech_id"], errors="ignore")
        new_df["batch_id"]  = pd.to_numeric(new_df["batch_id"], errors="coerce").fillna(0).astype(int)
        new_df["claim_id"] = pd.to_numeric(new_df["claim_id"], errors="ignore")
        new_df["unique_identifier_speech_batch_claim"] = new_df["unique_identifier_speech_batch_claim"]
        # Combine + drop duplicates on a sensible key set
        combined = pd.concat([master_df, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=UNIQUE_KEYS, keep="first")
        combined.to_csv(master_csv, index=False)
        master_df = combined
    else:
        # nothing new; keep existing
        pass

    return master_df

# --- Merge the resulting CSV with your results dataframe ---
def merge_with_results(master_csv: str, results) -> pd.DataFrame:
    """
    `results` is your original list of dicts ({'speech_id', 'batches', ...}) or a dataframe.
    Returns a merged dataframe on 'speech_id'.
    """
    master_df = pd.read_csv(master_csv)
    if isinstance(results, pd.DataFrame):
        results_df = results.copy()
    else:
        # assume list[dict]
        results_df = pd.DataFrame(results)
    

    
    results_df["batch_id"] = results_df.groupby("speech_id").cumcount() + 1

    # Keep only one row per speech in the metadata (if desired)
    # but preserve all columns you may want to carry over.
    if "speech_id" not in results_df.columns:
        raise ValueError("`results` must include a 'speech_id' column/field.")
    # Example: pick a subset of metadata columns if the DF is wide
    keep_cols = [c for c in results_df.columns ]
    #print(keep_cols)
    results_slim = results_df[keep_cols]#.drop_duplicates(subset=["speech_id"])
    #display(master_df)
    #display(results_df)
    # metadata df 
    
    merged = pd.merge(master_df,
         results_df,
        on = ['speech_id','batch_id'])

    merged = pd.merge(merged,df[['URL','Title',	'Subtitle',	'Date',	'Authorname','Role'	,'Gender',	'CentralBank',	'Country','Language','date','speech_id']],on='speech_id')
    merged.to_csv('../output/claims/merged_master_and_orginal.csv', index=False)
    return merged


In [350]:
# 1) Build/refresh the master CSV from all completed outputs in a folder
master_df = process_outputs_folder(
    folder="user_outputs/speech_runs",     # where your output_*.jsonl files are
    master_csv="claims_master.csv",
)

In [351]:
# 2) Merge with your original `results` (list of dicts) or a DataFrame
merged_master_and_orginal = merge_with_results("claims_master.csv", results_df)

# Example: inspect or save
#print(master_df.head())
#merged_df.to_csv("claims_master_merged.csv", index=False)

In [358]:
df[df['Authorname']=='Catherine L Mann']['speech_id'].tolist()

[16649, 16650, 16651, 16652, 16653, 16654, 16655]

In [361]:
merged_master_and_orginal.iloc[28:31]['batches'].values[0]

['Inflation rates have firmed around 10% in headline and 6% in core on a 3 month over 3 month annualized basis.',
 'There is no clear turning point in these UK data, which puts a premium on the other, more granular assessments of turning points.',
 'Monetary policy strategy Chart 9 shows how financial market-implied policy rate expectations were continuously revised through the forecast vintages, which in turn also affected the forecast, since these curves are a key conditioning assumption.',
 'The shifting peak in the market-implied curve, both in level space and timing of a peak in the hiking cycle, reflects the extent of uncertainty both around the evolution of data that underpins MPC decisions, as well as the MPCs reaction function.',
 'The shift up and steepness of the curve as well as its reversal reflect market expectations of the path that the MPC would need to take in order to meet its target.',
 'Chart 9: Bank of England policy rate and OIS curves over various forecast vintag

In [317]:
results_df[results_df['speech_id']==16652].shape

(11, 3)

In [321]:
master_df.shape

(118, 12)

In [320]:
pd.merge(master_df,
         results_df,
        on = ['speech_id','batch_id']).shape

(118, 13)

In [297]:
pd.merge(master_df,
         results_df[results_df['speech_id']==16652],
        on = ['speech_id','batch_id']).iloc[28:31]['batches'].values[0]

['Inflation rates have firmed around 10% in headline and 6% in core on a 3 month over 3 month annualized basis.',
 'There is no clear turning point in these UK data, which puts a premium on the other, more granular assessments of turning points.',
 'Monetary policy strategy Chart 9 shows how financial market-implied policy rate expectations were continuously revised through the forecast vintages, which in turn also affected the forecast, since these curves are a key conditioning assumption.',
 'The shifting peak in the market-implied curve, both in level space and timing of a peak in the hiking cycle, reflects the extent of uncertainty both around the evolution of data that underpins MPC decisions, as well as the MPCs reaction function.',
 'The shift up and steepness of the curve as well as its reversal reflect market expectations of the path that the MPC would need to take in order to meet its target.',
 'Chart 9: Bank of England policy rate and OIS curves over various forecast vintag

In [277]:
results[-1]['batches']

[['LAUNCH OF THE 2018 WORLD SAVINGS DAY THEME: SAVE, INVEST, AND SAVE: WHAT DO YOU WISH FOR? SPEECH BY DR.',
  'DENNY H. KALYALYA GOVERNOR - BANK OF ZAMBIA WEDNESDAY, 31st OCTOBER 2018 LUSAKA, ZAMBIA LAUNCH OF THE 2018 WORLD SAVINGS DAY, SPEECH BY DR.',
  'DENNY H. KALYALYA, GOVERNOR - BANK OF ZAMBIA - 31OCTOBER 2018 The Chief Executive - Securities and Exchange Commission; The Registrar - Pensions and Insurance Authority; The Chairperson - Bankers Association of Zambia; Senior Government Officials present; Members of the Diplomatic Corps; CEOs and Senior Private Sector Representatives; Representatives from the Universities; Members of the Media; Pupils and students here present; and Let me just say, Distinguished Ladies and Gentlemen.'],
 ['It is my honour and privilege to address you all this morning, at this years World Savings Day.',
  "As the Hon Minister of Finance indicated last evening, when she launched the event, this years global theme is:  What Do You Wish For? As a follow 

In [193]:


pd.concat([pd.DataFrame([pd.Series(i) for i in json.loads(parsed_results[index_]['content'])['claims']]) for index_ in range(len(parsed_results))])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [171]:
results[16652]['batches'][-3]

['What does the central banker need to do when faced with these different types of expectations formation? The model in Chart 9 shows the extent to which monetary policy needs to be increasingly restrictive to return the inflation rate to target when agents increasingly are backward-looking.',
 'We have to remember that the world has been hit by a sequence of large inflationary shocks, which have increased the risk of being in the purple world in which troubling __________________________________________________________________________________________________________________________ 14 It is also related to the Phillips multiplier of Barnichon & Meesters (2021) in that it attempts to capture the trade-off between inflation and economic activity over time.',
 'Bank of England Page 20 non-linearities are evident.',
 'I am not saying we are at that point yet or that we will necessarily get there given what we know now.',
 'However we need to be aware of how important the expectations form

In [112]:
len(parsed_results)

11

In [114]:
parsed_results[0]

{'custom_id': '16652_batch_1_be2160ab-83b3-4415-8a1c-a87ca5665723',
 'content': '\n{\n  "cause": "Central banks increase policy rates",\n  "effect": "Potential turning point in macroeconomic data",\n  "linking_explanation": "Central banks adjust policy rates in response to perceived changes in macroeconomic data, which may indicate a transition between phases of the business cycle.",\n  "document_reference": "Introduction, paragraph 1",\n  "quantifications": "Federal Reserve increased by 25 basis points, European Central Bank and Bank of England by 50 basis points.",\n  "uncertainty": "Medium"\n}\n\n\n',
 'parsed': {'cause': 'Central banks increase policy rates',
  'effect': 'Potential turning point in macroeconomic data',
  'linking_explanation': 'Central banks adjust policy rates in response to perceived changes in macroeconomic data, which may indicate a transition between phases of the business cycle.',
  'document_reference': 'Introduction, paragraph 1',
  'quantifications': 'Fede

# old code 

In [ ]:
import os
import json

# Updated response_format with detailed and standalone descriptions, rearranged to have full text fields first
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "causal_analysis_response_v13",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                # Full Text Fields
                # Research Questions
                "research_question_full_text": {
                    "type": "string",
                    "description": (
                        "From the **full text** of the paper, provide the research questions. If the research questions are explicitly stated in the full text, quote them verbatim and indicate that they are quoted. "
                        "If they are not explicitly stated, extract the research questions from the full text and mention that they are extracted."
                    )
                },
                # Causal Inference Information
                "causal_identification_full_text": {
                    "type": "string",
                    "description": (
                        "From the **full text** of the paper, provide an exhaustive and detailed summary including:\n"
                        "- Use of causal language and the overall level of tentativeness in the language used.\n"
                        "- For each causal claim made in the full text, including all intermediate steps, colliders, confounders, parents, children, ancestors, descendants, and any other relevant nodes in the causal graph (DAG), specify:\n"
                        "  - The causal relationships between variables.\n"
                        "  - The level of tentativeness and/or certainty in the language used (e.g., 'associates with', 'suggests', 'causes').\n"
                        "  - Identification strategies used with each claim (only if explicitly mentioned by the authors).\n"
                        "  - Sources of exogenous variation used for identification, if any.\n"
                        "  - Description of control groups used, if any.\n"
                        "- Indicate which of these relationships are examined with causal identification methods and which are explored with correlational analyses."
                    )
                },
                # Causal Claims
                "causal_claim_full_text": {
                    "type": "string",
                    "description": (
                        "From the **full text** of the paper, provide an exhaustive, detailed, and clear description of all causal claims made in the paper's narrative, capturing all intermediate steps, colliders, confounders, "
                        "parents, children, ancestors, descendants, and any other relevant nodes in the causal graph (DAG). The description should be in rich paragraph form and include:\n"
                        "- All causal relationships between variables, including multiple arrows from the same node and multiple source nodes pointing to the same sink node.\n"
                        "- For each causal claim, specify:\n"
                        "  - Source and target variables.\n"
                        "  - Any intermediate variables or mechanisms.\n"
                        "  - Type of causal relationship (e.g., direct effect, indirect effect, confounding, mediation).\n"
                        "  - Method used to establish the causal link (only if explicitly mentioned by the authors).\n"
                        "  - Effect size and direction (increase, decrease, or no effect), including magnitude if stated.\n"
                        "  - Statistical significance of the effect.\n"
                        "  - Null results and their statistical significance, if any.\n"
                        "  - Level of tentativeness in the language used.\n"
                        "- Indicate which relationships are examined with causal identification methods and which are explored with correlational analyses.\n"
                        "- Do not try to cut words in this section; be detailed, extensive, and exhaustive."
                    )
                },
                # Framing and Policy Implications
                "framing_and_policy_full_text": {
                    "type": "string",
                    "description": (
                        "From the **full text** of the paper, provide an information-rich overview of how the paper and its results are framed by the authors, including:\n"
                        "- Explicit policy recommendations made in the full text, if any.\n"
                        "- Entities to whom the recommendations are directed in the full text (e.g., national policymakers, international organizations, local governments, managers, firms).\n"
                        "- Key takeaways for researchers and non-researchers as presented by the authors in the full text.\n"
                        "- Focus only on explicit statements in the full text; do not interpret or infer implicit recommendations."
                    )
                },
                # Introduction Fields
                # Research Questions
                "research_question_intro": {
                    "type": "string",
                    "description": (
                        "From the **introduction** section of the paper, provide the research questions. If the research questions are explicitly stated in the introduction, quote them verbatim and indicate that they are quoted. "
                        "If they are not explicitly stated, extract the research questions from the introduction and mention that they are extracted."
                    )
                },
                # Causal Inference Information
                "causal_identification_intro": {
                    "type": "string",
                    "description": (
                        "From the **introduction** section of the paper, provide an exhaustive and detailed summary including:\n"
                        "- Use of causal language and the overall level of tentativeness in the language used.\n"
                        "- For each causal claim made in the introduction, including all intermediate steps, colliders, confounders, parents, children, ancestors, descendants, and any other relevant nodes in the causal graph (DAG), specify:\n"
                        "  - The causal relationships between variables.\n"
                        "  - The level of tentativeness and/or certainty in the language used.\n"
                        "  - Identification strategies used with each claim.\n"
                        "  - Sources of exogenous variation used for identification, if any.\n"
                        "  - Description of control groups used, if any.\n"
                        "- Indicate which of these relationships are examined with causal identification methods and which are explored with correlational analyses."
                    )
                },
                # Causal Claims
                "causal_claim_intro": {
                    "type": "string",
                    "description": (
                        "From the **introduction** section of the paper, provide an exhaustive, detailed, and clear description of all causal claims made in the paper's narrative, capturing all intermediate steps, colliders, confounders, "
                        "parents, children, ancestors, descendants, and any other relevant nodes in the causal graph (DAG). The description should be in rich paragraph form and include:\n"
                        "- All causal relationships between variables, including multiple arrows from the same node and multiple source nodes pointing to the same sink node.\n"
                        "- For each causal claim, specify:\n"
                        "  - Source and target variables.\n"
                        "  - Any intermediate variables or mechanisms.\n"
                        "  - Type of causal relationship.\n"
                        "  - Method used to establish the causal link (only if explicitly mentioned by the authors).\n"
                        "  - Effect size and direction, including magnitude if stated.\n"
                        "  - Statistical significance of the effect.\n"
                        "  - Null results and their statistical significance, if any.\n"
                        "  - Level of tentativeness in the language used.\n"
                        "- Indicate which relationships are examined with causal identification methods and which are explored with correlational analyses.\n"
                        "- Do not try to cut words in this section; be detailed, extensive, and exhaustive."
                    )
                },
                # Framing and Policy Implications
                "framing_and_policy_intro": {
                    "type": "string",
                    "description": (
                        "From the **introduction** section of the paper, provide an information-rich overview of how the paper and its results are framed by the authors, including:\n"
                        "- Explicit policy recommendations made in the introduction, if any.\n"
                        "- Entities to whom the recommendations are directed in the introduction (e.g., national policymakers, international organizations, local governments, managers, firms).\n"
                        "- Key takeaways for researchers and non-researchers as presented by the authors in the introduction.\n"
                        "- Focus only on explicit statements in the introduction; do not interpret or infer implicit recommendations."
                    )
                },
                # Abstract Fields
                # Research Questions
                "research_question_abstract": {
                    "type": "string",
                    "description": (
                        "From the **abstract** of the paper, provide the research questions. If the research questions are explicitly stated in the abstract, quote them verbatim and indicate that they are quoted. "
                        "If they are not explicitly stated, extract the research questions from the abstract and mention that they are extracted."
                    )
                },
                # Causal Inference Information
                "causal_identification_abstract": {
                    "type": "string",
                    "description": (
                        "From the **abstract** of the paper, provide an exhaustive and detailed summary including:\n"
                        "- Use of causal language and the overall level of tentativeness in the language used.\n"
                        "- For each causal claim made in the abstract, including all intermediate steps, colliders, confounders, parents, children, ancestors, descendants, and any other relevant nodes in the causal graph (DAG), specify:\n"
                        "  - The causal relationships between variables.\n"
                        "  - The level of tentativeness and/or certainty in the language used.\n"
                        "  - Identification strategies used with each claim (only if explicitly mentioned by the authors).\n"
                        "  - Sources of exogenous variation used for identification, if any.\n"
                        "  - Description of control groups used, if any.\n"
                        "- Indicate which of these relationships are examined with causal identification methods and which are explored with correlational analyses."
                    )
                },
                # Causal Claims
                "causal_claim_abstract": {
                    "type": "string",
                    "description": (
                        "From the **abstract** of the paper, provide an exhaustive, detailed, and clear description of all causal claims made in the paper's narrative, capturing all intermediate steps, colliders, confounders, "
                        "parents, children, ancestors, descendants, and any other relevant nodes in the causal graph (DAG). The description should be in rich paragraph form and include:\n"
                        "- All causal relationships between variables, including multiple arrows from the same node and multiple source nodes pointing to the same sink node.\n"
                        "- For each causal claim, specify:\n"
                        "  - Source and target variables.\n"
                        "  - Any intermediate variables or mechanisms.\n"
                        "  - Type of causal relationship.\n"
                        "  - Method used to establish the causal link (only if explicitly mentioned by the authors).\n"
                        "  - Effect size and direction, including magnitude if stated.\n"
                        "  - Statistical significance of the effect.\n"
                        "  - Null results and their statistical significance, if any.\n"
                        "  - Level of tentativeness in the language used.\n"
                        "- Indicate which relationships are examined with causal identification methods and which are explored with correlational analyses.\n"
                        "- Do not try to cut words in this section; be detailed, extensive, and exhaustive."
                    )
                },
                # Framing and Policy Implications
                "framing_and_policy_abstract": {
                    "type": "string",
                    "description": (
                        "From the **abstract** of the paper, provide an information-rich overview of how the paper and its results are framed by the authors, including:\n"
                        "- Explicit policy recommendations made in the abstract, if any.\n"
                        "- Entities to whom the recommendations are directed in the abstract (e.g., national policymakers, international organizations, local governments, managers, firms).\n"
                        "- Key takeaways for researchers and non-researchers as presented by the authors in the abstract.\n"
                        "- Focus only on explicit statements in the abstract; do not interpret or infer implicit recommendations."
                    )
                },
                # General Information
                "authors": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of authors' names."
                },
                "authors_institutions": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Primary university affiliations of the authors, in the same order as the authors."
                },
                "authors_emails": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Email addresses of the authors, if available."
                },
                "all_institutions": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Unique list of all institutions affiliated with the authors."
                },
                "title": {
                    "type": "string",
                    "description": "Title of the paper."
                },
                "year_of_release": {
                    "type": "integer",
                    "description": "Year the paper was released."
                },
                "jel_codes": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Mentioned JEL codes, if any."
                },
                "keywords": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Mentioned keywords, if any."
                },
                # Fields
                "is_finance": {
                    "type": "boolean",
                    "description": "True if the paper relates to Finance; otherwise, false."
                },
                "is_development": {
                    "type": "boolean",
                    "description": "True if the paper relates to Development; otherwise, false."
                },
                "is_labour": {
                    "type": "boolean",
                    "description": "True if the paper relates to Labour; otherwise, false."
                },
                "is_public_economics": {
                    "type": "boolean",
                    "description": "True if the paper relates to Public Economics; otherwise, false."
                },
                "is_urban_economics": {
                    "type": "boolean",
                    "description": "True if the paper relates to Urban Economics; otherwise, false."
                },
                "is_macroeconomics": {
                    "type": "boolean",
                    "description": "True if the paper relates to Macroeconomics; otherwise, false."
                },
                "is_behavioral_economics": {
                    "type": "boolean",
                    "description": "True if the paper relates to Behavioral Economics; otherwise, false."
                },
                "is_economic_history": {
                    "type": "boolean",
                    "description": "True if the paper relates to Economic History; otherwise, false."
                },
                "is_econometric_theory": {
                    "type": "boolean",
                    "description": "True if the paper relates to Econometric Theory; otherwise, false."
                },
                "is_industrial_organization": {
                    "type": "boolean",
                    "description": "True if the paper relates to Industrial Organization; otherwise, false."
                },
                "is_environmental_economics": {
                    "type": "boolean",
                    "description": "True if the paper relates to Environmental Economics; otherwise, false."
                },
                "is_health_economics": {
                    "type": "boolean",
                    "description": "True if the paper relates to Health Economics; otherwise, false."
                },
                # Classification of Paper
                "classification_of_paper": {
                    "type": "string",
                    "enum": ["is_empirical_only", "mostly_empirical", "mostly_theoretical", "is_theoretical_only"],
                    "description": (
                        "Classification based on empirical and theoretical content of the paper:\n"
                        "- 'is_empirical_only': The paper is entirely empirical with no theoretical content.\n"
                        "- 'mostly_empirical': The paper is primarily empirical with some theoretical content.\n"
                        "- 'mostly_theoretical': The paper is primarily theoretical with some empirical content.\n"
                        "- 'is_theoretical_only': The paper is entirely theoretical with no empirical content."
                    )
                },
                # Data and Units of Analysis
                "data_and_units_of_analysis": {
                    "type": "string",
                    "description": (
                        "Provide an information-rich summary including the following details:\n"
                        "- Total number of observations used in the analyses. Mention if there are multiple sets of analyses.\n"
                        "- Number of unique units analyzed (e.g., individuals, firms) in each of the analyses.\n"
                        "- Unit of analysis (e.g., individual, firm, sector) in each of the analyses.\n"
                        "- Data granularity (e.g., micro-level, macro-level) in each of the analyses.\n"
                        "- Data nomenclatures or classification systems used for each of the datasets (e.g., SIC codes, HS codes).\n"
                        "- Temporal context of the data, e.g., years.\n"
                        "- Geographical context of the data with country codes in ISO 3166-1 alpha-3 format.\n"
                        "- Any other contextual information relevant to the datasets."
                    )
                },
                # Data Accessibility
                "data_accessibility": {
                    "type": "string",
                    "description": (
                        "Provide a detailed summary covering all of the following aspects of all of the data used:\n"
                        "- Ownership: Identify who owns the data (e.g., private company, public sector entity, researchers).\n"
                        "- Ownership detail: Provide more details on who owns which data and what are the names of the relevant organizations, if mentioned.\n"
                        "- Accessibility: State whether the data is freely accessible or restricted.\n"
                        "- Access Method: Explain how the researchers accessed the data (e.g., public databases, special agreements).\n"
                        "- Informed Consent: Indicate whether researchers obtained explicit informed consent from participants.\n"
                        "- IRB Approval: Mention any Institutional Review Board (IRB) approval or ethical considerations.\n"
                        "- Data Privacy: Discuss any data privacy, confidentiality, or data protection measures mentioned.\n"
                        "- Access Instructions: Describe how others can access the data (e.g., available upon request, proprietary restrictions).\n"
                        "- Confidentiality: Specify the level of data confidentiality (e.g., anonymized, sensitive data).\n"
                        "- Data Sharing: Note any data sharing policies or limitations mentioned."
                    )
                },
                # Institutional and Author-Level Information
                "publication_outlet": {
                    "type": "string",
                    "description": "Full name of the journal where the paper is published or intended to be published. If not available, write 'NA'."
                },
                "date_of_publication": {
                    "type": "string",
                    "description": "Date of publication in DD/MM/YYYY format. If not available, write 'NA'."
                },
                # Acknowledgement, Gratitude and Funders
                "acknowledgement": {
                    "type": "string",
                    "description": (
                        "Full text summary of the acknowledgements section of the paper, typically located on the first page or in the footnotes. "
                        "This summary should include the following:\n"
                        "- **Gratitude:** Names of individuals or organizations explicitly thanked by the authors and the reasons for their acknowledgement. "
                        "Reasons may include providing feedback, helpful comments, research assistance (RA), or other contributions.\n"
                        "- **Funders:** Names of the funding agencies, grants, and associated grant numbers. Ensure to list the full names of the funding bodies "
                        "or institutions and include any available grant or project numbers.\n"
                        "If no acknowledgements or funders are mentioned, write 'NA' for the respective sections."
                    )
                }
            },
            "required": [
                # Full Text Fields
                "research_question_full_text",
                "causal_identification_full_text",
                "causal_claim_full_text",
                "framing_and_policy_full_text",
                # Introduction Fields
                "research_question_intro",
                "causal_identification_intro",
                "causal_claim_intro",
                "framing_and_policy_intro",
                # Abstract Fields
                "research_question_abstract",
                "causal_identification_abstract",
                "causal_claim_abstract",
                "framing_and_policy_abstract",
                # General Information
                "authors", "authors_institutions", "authors_emails", "all_institutions",
                "title", "year_of_release", "jel_codes", "keywords",
                "is_finance", "is_development", "is_labour", "is_public_economics",
                "is_urban_economics", "is_macroeconomics", "is_behavioral_economics",
                "is_economic_history", "is_econometric_theory", "is_industrial_organization",
                "is_environmental_economics", "is_health_economics",
                "classification_of_paper",
                # Data and Units of Analysis
                "data_and_units_of_analysis",
                # Data Accessibility
                "data_accessibility",
                # Institutional and Author-Level Information
                "publication_outlet", "date_of_publication",
                # Acknowledgement
                "acknowledgement"
            ],
            "additionalProperties": False
        }
    }
}

def create_jsonl_entry(file_path, label):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
        return {
            "custom_id": os.path.basename(file_path).replace('.txt', '') + '_' + label,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-5",
                "messages": [
                    {
                        "role": "system",
                        "content": (
                            "You are an expert assistant specializing in analyzing economics papers. Your task is to analyze the text "
                            "content of a provided economics paper and extract specific information related to metadata, research questions, "
                            "causal claims, research methods, variables, data measurements, and identification strategies.\n\n"
                            "Please note that you are given only the first 30 pages of the paper, so some information may not be available.\n\n"
                            "**Important Instructions:**\n"
                            "- Provide clear, detailed, and information-rich responses for each field as specified below.\n"
                            "- For fields requiring free text responses, use information-dense language to maximize content while minimizing wordiness.\n"
                            "- Include both general descriptions and specific details as requested for each field.\n"
                            "- Focus exclusively on the specified section of the paper when extracting information (**full text**, **introduction**, or **abstract**). When a field requires information only from a specific section, ensure you extract only from that section and mention it explicitly.\n"
                            "- Only assign methods or identification strategies if they are explicitly mentioned by the authors.\n"
                            "- **Causal Claims Extraction:**\n"
                            "  - Extract an exhaustive and detailed description of each causal claim made in the paper's narrative, including all intermediate steps, colliders, confounders, parents, children, ancestors, descendants, and any other relevant nodes in the causal graph (DAG).\n"
                            "  - Allow for multiple arrows pointing from the same nodes and multiple source nodes pointing to the same sink node.\n"
                            "  - Do not try to cut words in this section; be detailed, extensive, and exhaustive.\n"
                            "  - For each causal claim, specify which relationships are examined with causal identification methods and which are explored with correlational analyses.\n"
                            "- Present the causal claims in rich paragraph form, capturing all details needed to extract the structured information downstream.\n"
                            "- For each causal claim, include the method used, effect size, direction, statistical significance, null results, and language tentativeness.\n"
                            "- Ensure all causal claims are presented in the order they appear in the paper's narrative.\n"
                            "- If information is not available, indicate it as 'NA'.\n"
                            "- Assess the level of tentativeness in the language used by the authors when presenting each causal claim.\n"
                            "- Do not interpret implicit recommendations; focus only on how the authors have presented their work explicitly.\n"
                            "- Be consistent in formatting to aid downstream processing.\n"
                            "- Be thorough and err on the side of over-extracting information rather than under-extracting.\n\n"
                            "**Fields to Extract:**\n\n"
                            "1. **Research Questions from Full Text:**\n"
                            "- **research_question_full_text**: From the **full text** of the paper, provide the research questions. If the research questions are explicitly stated in the full text, quote them verbatim and indicate that they are quoted. If they are not explicitly stated, extract the research questions from the full text and mention that they are extracted.\n\n"
                            "2. **Causal Inference Information from Full Text:**\n"
                            "- **causal_identification_full_text**: From the **full text** of the paper, provide an exhaustive and detailed summary including all elements as specified.\n\n"
                            "3. **Causal Claims from Full Text:**\n"
                            "- **causal_claim_full_text**: From the **full text** of the paper, provide an exhaustive, detailed, and clear description of all causal claims as specified.\n\n"
                            "4. **Framing and Policy Implications from Full Text:**\n"
                            "- **framing_and_policy_full_text**: From the **full text** of the paper, provide an information-rich overview as specified.\n\n"
                            "5. **Research Questions from Introduction:**\n"
                            "- **research_question_intro**: From the **introduction** section of the paper, provide the research questions as specified.\n\n"
                            "6. **Causal Inference Information from Introduction:**\n"
                            "- **causal_identification_intro**: From the **introduction** section of the paper, provide an exhaustive and detailed summary as specified.\n\n"
                            "7. **Causal Claims from Introduction:**\n"
                            "- **causal_claim_intro**: From the **introduction** section of the paper, provide an exhaustive, detailed, and clear description of all causal claims as specified.\n\n"
                            "8. **Framing and Policy Implications from Introduction:**\n"
                            "- **framing_and_policy_intro**: From the **introduction** section of the paper, provide an information-rich overview as specified.\n\n"
                            "9. **Research Questions from Abstract:**\n"
                            "- **research_question_abstract**: From the **abstract** of the paper, provide the research questions as specified.\n\n"
                            "10. **Causal Inference Information from Abstract:**\n"
                            "- **causal_identification_abstract**: From the **abstract** of the paper, provide an exhaustive and detailed summary as specified.\n\n"
                            "11. **Causal Claims from Abstract:**\n"
                            "- **causal_claim_abstract**: From the **abstract** of the paper, provide an exhaustive, detailed, and clear description of all causal claims as specified.\n\n"
                            "12. **Framing and Policy Implications from Abstract:**\n"
                            "- **framing_and_policy_abstract**: From the **abstract** of the paper, provide an information-rich overview as specified.\n\n"
                            "13. **General Information:**\n"
                            "- **Authors:** List of authors' names.\n"
                            "- **Authors' Institutions:** Primary university affiliations of the authors, in the same order as the authors.\n"
                            "- **Authors' Emails:** Email addresses of the authors, if available.\n"
                            "- **All Institutions:** Unique list of all institutions affiliated with the authors.\n"
                            "- **Title:** Title of the paper.\n"
                            "- **Year of Release:** Year the paper was released.\n"
                            "- **JEL Codes:** Mentioned JEL codes, if any.\n"
                            "- **Keywords:** Mentioned keywords, if any.\n"
                            "- **Fields (binary flags):** Indicate 'true' or 'false' for each of the following fields:\n"
                            "  - Finance, Development, Labour, Public Economics, Urban Economics, Macroeconomics,\n"
                            "    Behavioral Economics, Economic History, Econometric Theory, Industrial Organization,\n"
                            "    Environmental Economics, Health Economics.\n"
                            "- **Classification of Paper:** Choose one of:\n"
                            "  - 'is_empirical_only', 'mostly_empirical', 'mostly_theoretical', 'is_theoretical_only'.\n\n"
                            "14. **Data and Units of Analysis:**\n"
                            "- **data_and_units_of_analysis**: Provide an information-rich summary including all details as specified.\n\n"
                            "15. **Data Accessibility:**\n"
                            "- **data_accessibility**: Provide a detailed summary covering all aspects as specified.\n\n"
                            "16. **Institutional and Author-Level Information:**\n"
                            "- **Publication Outlet:** Full name of the journal where the paper is published or intended to be published. If not available, write 'NA'.\n"
                            "- **Date of Publication:** Date in DD/MM/YYYY format. If not available, write 'NA'.\n\n"
                            "17. **Acknowledgement, Gratitude and Funders:**\n"
                            "- **acknowledgement**: Extract the full-text summary of the acknowledgements, including gratitude and funders, as specified.\n\n"
                            "**Notes:**\n"
                            "- Ensure clarity, accuracy, detail, information-density, and thoroughness in your responses.\n"
                            "- Adhere strictly to specified formats and instructions.\n"
                            "- Use the canonical definitions provided.\n"
                            "- Information should be suitable for downstream processing by another language model.\n"
                            "- Focus exclusively on explicit content from the specified sections of the paper; avoid adding interpretations or assumptions.\n"
                            "- Be consistent in formatting and presentation to aid in downstream processing."
                        )
                    },
                    {
                        "role": "user",
                        "content": f"Here is the text extract of an economics paper:\n\n{text}"
                    }
                ],
                "temperature": 0.5,
                "max_completion_tokens": 16000,
                "response_format": response_format  # Use the updated response format
            }
        }


In [ ]:

# List to store file paths and labels
files_to_process = []

for dir_path, label in [(nber_dir, 'nber'), (cepr_dir, 'cepr')]:
    files = [f for f in os.listdir(dir_path) if f.endswith('.txt')]
    for filename in files:
        file_path = os.path.join(dir_path, filename)
        files_to_process.append((file_path, label))

# Create the JSONL file for batch processing
jsonl_filename = "batch_input_v7_full.jsonl"
with open(jsonl_filename, 'w', encoding='utf-8') as jsonl_file:
    for file_path, label in files_to_process:
        entry = create_jsonl_entry(file_path, label)
        jsonl_file.write(json.dumps(entry) + '\n')

print("JSONL file created successfully.")


JSONL file created successfully.


deploy on full

# Batches to be in <100mb chunks. Let's split them

Step 1: Split the JSONL File into N Batches

In [ ]:
# import os
# import pandas as pd
# import json

# # Define the directories
# nber_dir = "../../../2_raw_data/1_nber/nber_pdf_text_all_processed_filt/"
# cepr_dir = "../../../2_raw_data/2_cepr/cepr_pdf_text_all_processed_filt/"

# # Read the CSV file, ensuring paper_id is read as a string
# paper_ids_df = pd.read_csv("paper_ids_v7.csv", dtype={"paper_id": str})

# # Create a dictionary for easy lookup
# paper_ids_dict = paper_ids_df.groupby('paper_repo')['paper_id'].apply(set).to_dict()



In [ ]:

# # List to store file paths and labels
# files_to_process = []

# # Process the files in each directory based on paper_repo
# for dir_path, label in [(nber_dir, 'nber'), (cepr_dir, 'cepr')]:
#     files = [f for f in os.listdir(dir_path) if f.endswith('.txt')]
    
#     # Filter the files based on paper_ids from the CSV
#     valid_ids = paper_ids_dict.get(label, set())
#     for filename in files:
#         paper_id = filename.split('.')[0]  # Assuming filenames are paper_id.txt
#         if paper_id in valid_ids:
#             file_path = os.path.join(dir_path, filename)
#             files_to_process.append((file_path, label))

# # Create the JSONL file for batch processing
# jsonl_filename = "batch_input_v7_full.jsonl"
# with open(jsonl_filename, 'w', encoding='utf-8') as jsonl_file:
#     for file_path, label in files_to_process:
#         entry = create_jsonl_entry(file_path, label)  # Assuming this function is defined elsewhere
#         jsonl_file.write(json.dumps(entry) + '\n')

# print("JSONL file created successfully.")


JSONL file created successfully.


In [ ]:
import os
import json

# create input_batches directory if its doesnt exist
if not os.path.exists("../Data/Processed/batches_of_speeches"):
    os.makedirs("../Data/Processed/batches_of_speeches")

def split_speech(num_batches):



# Split JSONL file into smaller chunks
def split_jsonl_file(file_path, num_batches):
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    total_lines = len(lines)
    lines_per_batch = total_lines // num_batches
    
    for i in range(num_batches):
        batch_lines = lines[i * lines_per_batch : (i + 1) * lines_per_batch]
        batch_file_path = f"input_batches/batch_input_v7_part{i + 1}.jsonl"
        with open(batch_file_path, 'w', encoding='utf-8') as batch_file:
            batch_file.writelines(batch_lines)
    
    # If there are leftover lines, add them to the last batch
    if total_lines % num_batches != 0:
        with open(f"input_batches/batch_input_v7_part{num_batches}.jsonl", 'a', encoding='utf-8') as batch_file:
            batch_file.writelines(lines[num_batches * lines_per_batch :])
    
    print("Batches created successfully.")

# Run the function
split_jsonl_file("batch_input_v7_full.jsonl", 50)


IndentationError: expected an indented block after function definition on line 8 (2296020042.py, line 13)

In [ ]:
import os
import json
import random

# Function to create a sample JSONL file with 1 random lines
def create_sample_jsonl(file_path, sample_size=1):
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    # Randomly select 10 lines from the full file
    sample_lines = random.sample(lines, sample_size)
    
    # Create a smaller JSONL file with just the sample lines
    sample_file_path = "sample_input_v7.jsonl"
    with open(sample_file_path, 'w', encoding='utf-8') as sample_file:
        sample_file.writelines(sample_lines)
    
    print(f"Sample of {sample_size} lines created successfully in {sample_file_path}.")
    return sample_file_path

# Function to upload the sample file and create a batch
def upload_and_create_sample_batch(sample_file_path):
    # Initialize OpenAI client
    client = OpenAI()

    # Upload the sample JSONL file
    sample_input_file = client.files.create(
        file=open(sample_file_path, "rb"),
        purpose="batch"
    )
    
    sample_input_file_id = sample_input_file.id
    
    # Create a batch with the sample file
    batch = client.batches.create(
        input_file_id=sample_input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={
            "description": "Pilot Economics paper analysis with 10 random lines"
        }
    )
    
    print(f"Sample batch job created successfully with Batch ID: {batch.id}")
    return batch.id

# Run the process
sample_file = create_sample_jsonl("batch_input_v7_full.jsonl", 1)
sample_batch_id = upload_and_create_sample_batch(sample_file)


Sample of 1 lines created successfully in sample_input_v7.jsonl.
Sample batch job created successfully with Batch ID: batch_6706b9036bd08190bf7b1b51722a3f60


Step 2: Upload Each Batch for Processing

In [ ]:

import time
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()

# Function to upload and create batches
def upload_and_create_batches(num_batches):
    batch_ids = []
    for i in range(1, num_batches + 1):
        batch_file_path = f"input_batches/batch_input_v7_part{i}.jsonl"
        
        # Upload the JSONL file for batch processing
        batch_input_file = client.files.create(
            file=open(batch_file_path, "rb"),
            purpose="batch"
        )
        
        batch_input_file_id = batch_input_file.id
        
        # Create the batch
        batch = client.batches.create(
            input_file_id=batch_input_file_id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={
                "description": f"Economics paper analysis job v7 part {i}"
            }
        )
        
        batch_ids.append(batch.id)
        print(f"Batch {i} job created successfully with Batch ID: {batch.id}")
        
        # Pause between uploads to avoid hitting rate limits
        time.sleep(1)
    
    return batch_ids

# Run the function
batch_ids = upload_and_create_batches(50)


KeyboardInterrupt: 

Step 3: Check the Status of Each Batch and Retrieve the Results

In [ ]:
batch_ids = [
    'batch_6706baed7d9881908c52c285f489f825',
    'batch_6706bb2225fc8190962a931136e18372',
    'batch_6706bb4e95dc819099d4628e3db38740',
    'batch_6706bb84a3e08190a03cf44ed965beb2',
    'batch_6706bbb93cf481908ff364a8803698a5',
    'batch_6706bbfd4af48190805fb6f5b1b430bd',
    'batch_6706bc45413481909cbea7622cac983d',
    'batch_6706bc91614c8190a60dbba76e3a52b6',
    'batch_6706bcdad4888190910f160cc5d59095',
    'batch_6706bd272d2881909dd8cf06446f1d7d',
    'batch_6706bd77b8288190b9b8b16722c09f84',
    'batch_6706bdc942d08190b179801f935054b9',
    'batch_6706be12de5c8190bd4179a12a36e966',
    'batch_6706be6a149c8190be3a311324aaa53a',
    'batch_6706bec1e5e881909d7d54c785ee7bf4',
    'batch_6706bf07753c819081fa6a7445100f26',
    'batch_6706bf32c44c81908b4b7e4dbd738d0d',
    'batch_6706bf5e7f8081909c9a6923820f54ea',
    'batch_6706bf89a37881909adf2f057b5e1471',
    'batch_6706bfb4dce88190b3f0293fa156db9b',
    'batch_6706bfe0b8d88190a109c22d2db7e7ff',
    'batch_6706c00c55f081909b7339f7cfb8b891',
    'batch_6706c0280d8881909c569291289f67ec',
    'batch_6706c040ba3481908aca575d99b9f912',
    'batch_6706c059d694819097adc428ff4de561',
    'batch_6706c071bd488190b70667778d94e996',
    'batch_6706c088ba9481908bd5fe691f83b232',
    'batch_6706c09da3408190865c3121f377ed5d',
    'batch_6706c0b2a138819096a8266bef5e831e',
    'batch_6706c0c7cfc88190aad3a6973d68639b',
    'batch_6706c0dd229481909ea3ae1fc6098338',
    'batch_6706c0f5bb4c81909cd58ccb2f90146e',
    'batch_6706c12159308190afb132798ae46d64',
    'batch_6706c14c61308190b4ddaeb4b04262e1',
    'batch_6706c178d11c819093786d46ae0f196f',
    'batch_6706c1a4de4c819095010b94dc382718',
    'batch_6706c1d0384481909be75e0fabbe8ec7',
    'batch_6706c1fc100081909f547053be2dc5b2',
    'batch_6706c227e1748190a7d809a996f558da',
    'batch_6706c254702c8190bac8771d43578849',
    'batch_6706c27eb76c819091971e182616583c',
    'batch_6706c2a5d25481909dd98384f2b45b4c',
    'batch_6706c2cd65f08190be32850c1c26e78f',
    'batch_6706c2f517808190a0dc8064bbcb4811',
    'batch_6706c31e35488190bdbb653928cb1979',
    'batch_6706c34824348190bc1ec0da9202ee32',
    'batch_6706c37116648190825727d07a20e51e',
    'batch_6706c39b1d1c8190b270f0f6adb5ab40',
    'batch_6706c3c4acac8190bfe6d7a2915b91b6',
    'batch_6706c3f149908190ac8d0cd255adf5ca'
]

In [ ]:
# create output_batches directory if not existing
if not os.path.exists("output_batches"):
    os.makedirs("output_batches")

# Function to check the status of the batches
def check_batch_status(batch_ids):
    for batch_id in batch_ids:
        batch_status = client.batches.retrieve(batch_id)
        print(f"Status for Batch ID {batch_id}: {batch_status.status}")
        if batch_status.status == 'completed':
            # Retrieve the output file
            output_file_id = batch_status.output_file_id
            file_response = client.files.content(output_file_id)
            output_file_path = f"output_batches/batch_output_v7_{batch_id}.jsonl"
            with open(output_file_path, "w", encoding='utf-8') as output_file:
                output_file.write(file_response.text)
            print(f"Batch output saved to {output_file_path}")
        else:
            print("Batch not completed yet. Please check again later.")

# Example usage: Check the status of the batches
check_batch_status(batch_ids)


Status for Batch ID batch_6706baed7d9881908c52c285f489f825: completed
Batch output saved to output_batches/batch_output_v7_batch_6706baed7d9881908c52c285f489f825.jsonl
Status for Batch ID batch_6706bb2225fc8190962a931136e18372: completed
Batch output saved to output_batches/batch_output_v7_batch_6706bb2225fc8190962a931136e18372.jsonl
Status for Batch ID batch_6706bb4e95dc819099d4628e3db38740: completed
Batch output saved to output_batches/batch_output_v7_batch_6706bb4e95dc819099d4628e3db38740.jsonl
Status for Batch ID batch_6706bb84a3e08190a03cf44ed965beb2: completed
Batch output saved to output_batches/batch_output_v7_batch_6706bb84a3e08190a03cf44ed965beb2.jsonl
Status for Batch ID batch_6706bbb93cf481908ff364a8803698a5: completed
Batch output saved to output_batches/batch_output_v7_batch_6706bbb93cf481908ff364a8803698a5.jsonl
Status for Batch ID batch_6706bbfd4af48190805fb6f5b1b430bd: completed
Batch output saved to output_batches/batch_output_v7_batch_6706bbfd4af48190805fb6f5b1b430b

Step 4: Combine the Results from All Batches into a Single DataFrame and Save as CSV

In [ ]:
import os
import json
import pandas as pd
import ast
import logging
from tqdm import tqdm

# Configure logging
logging.basicConfig(filename='parsing_errors.log', level=logging.ERROR,
                    format='%(asctime)s:%(levelname)s:%(message)s')

# Directory where your batch JSONL files are stored
batch_files_dir = 'output_batches/'  # Adjust this to the directory where your batches are located

# Function to load a JSONL file into a list of dictionaries
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [json.loads(line) for line in file.readlines()]

# Updated function to process and normalize the new response format
def normalize_json(data):
    flat_data = []
    for entry in data:
        # Check if the entry has a valid response structure
        if (
            'response' in entry
            and 'body' in entry['response']
            and 'choices' in entry['response']['body']
        ):
            custom_id = entry.get('custom_id', 'unknown_id')  # Store the custom_id here
            for choice in entry['response']['body']['choices']:
                message_content = choice['message']['content']
                try:
                    # Parse the JSON content from the message
                    content_json = json.loads(message_content)
                    content_json['custom_id'] = custom_id  # Add custom_id to the content

                    # Initialize a dictionary to hold the flattened data for this paper
                    paper_entry = {}

                    # 1. **Full Text Fields**
                    paper_entry['research_question_full_text'] = content_json.get('research_question_full_text', 'NA')
                    paper_entry['causal_identification_full_text'] = content_json.get('causal_identification_full_text', 'NA')
                    paper_entry['causal_claim_full_text'] = content_json.get('causal_claim_full_text', 'NA')
                    paper_entry['framing_and_policy_full_text'] = content_json.get('framing_and_policy_full_text', 'NA')

                    # 2. **Introduction Fields**
                    paper_entry['research_question_intro'] = content_json.get('research_question_intro', 'NA')
                    paper_entry['causal_identification_intro'] = content_json.get('causal_identification_intro', 'NA')
                    paper_entry['causal_claim_intro'] = content_json.get('causal_claim_intro', 'NA')
                    paper_entry['framing_and_policy_intro'] = content_json.get('framing_and_policy_intro', 'NA')

                    # 3. **Abstract Fields**
                    paper_entry['research_question_abstract'] = content_json.get('research_question_abstract', 'NA')
                    paper_entry['causal_identification_abstract'] = content_json.get('causal_identification_abstract', 'NA')
                    paper_entry['causal_claim_abstract'] = content_json.get('causal_claim_abstract', 'NA')
                    paper_entry['framing_and_policy_abstract'] = content_json.get('framing_and_policy_abstract', 'NA')

                    # 4. **General Information**
                    # Handle array fields by joining them with semicolons
                    array_fields = ['authors', 'authors_institutions', 'authors_emails', 'all_institutions', 'jel_codes', 'keywords']
                    for field in array_fields:
                        value = content_json.get(field, 'NA')
                        if isinstance(value, list):
                            paper_entry[field] = '; '.join(map(str, value))
                        else:
                            paper_entry[field] = value if value else 'NA'

                    # Handle string fields
                    string_fields = ['title']
                    for field in string_fields:
                        paper_entry[field] = content_json.get(field, 'NA')

                    # Handle integer fields
                    paper_entry['year_of_release'] = content_json.get('year_of_release', 'NA')
                    if isinstance(paper_entry['year_of_release'], int):
                        pass  # Keep as integer
                    else:
                        try:
                            paper_entry['year_of_release'] = int(paper_entry['year_of_release'])
                        except (ValueError, TypeError):
                            paper_entry['year_of_release'] = 'NA'

                    # 5. **Fields (Binary Flags)**
                    boolean_fields = [
                        'is_finance', 'is_development', 'is_labour', 'is_public_economics',
                        'is_urban_economics', 'is_macroeconomics', 'is_behavioral_economics',
                        'is_economic_history', 'is_econometric_theory', 'is_industrial_organization',
                        'is_environmental_economics', 'is_health_economics'
                    ]
                    for field in boolean_fields:
                        value = content_json.get(field, False)
                        if isinstance(value, bool):
                            paper_entry[field] = value
                        else:
                            # Handle cases where the value might not be boolean
                            paper_entry[field] = True if str(value).lower() in ['true', '1'] else False

                    # 6. **Classification of Paper**
                    classification_options = ["is_empirical_only", "mostly_empirical", "mostly_theoretical", "is_theoretical_only"]
                    classification = content_json.get('classification_of_paper', 'NA')
                    if classification not in classification_options:
                        classification = 'NA'
                    paper_entry['classification_of_paper'] = classification

                    # 7. **Data and Units of Analysis**
                    paper_entry['data_and_units_of_analysis'] = content_json.get('data_and_units_of_analysis', 'NA')

                    # 8. **Data Accessibility**
                    paper_entry['data_accessibility'] = content_json.get('data_accessibility', 'NA')

                    # 9. **Institutional and Author-Level Information**
                    paper_entry['publication_outlet'] = content_json.get('publication_outlet', 'NA')
                    paper_entry['date_of_publication'] = content_json.get('date_of_publication', 'NA')

                    # 10. **Acknowledgement, Gratitude and Funders**
                    paper_entry['acknowledgement'] = content_json.get('acknowledgement', 'NA')

                    # Add custom_id to the flattened entry
                    paper_entry['custom_id'] = custom_id

                    # Append the paper_entry to flat_data
                    flat_data.append(paper_entry)

                except (json.JSONDecodeError, TypeError) as e:
                    logging.error(f"Failed to parse content for custom_id {custom_id}. Error: {e}")
                    logging.error(f"Content: {message_content}")
        else:
            logging.error(f"No valid response found in entry with custom_id {entry.get('custom_id', 'unknown')}")

    return pd.DataFrame(flat_data)

# Function to load all batch files and create a DataFrame
def load_and_parse_batches(batch_files_dir):
    batch_files = [
        os.path.join(batch_files_dir, f)
        for f in os.listdir(batch_files_dir)
        if f.endswith('.jsonl')
    ]
    all_data = []

    # Load and process each batch file with progress tracking
    for batch_file in tqdm(batch_files, desc="Processing batch files"):
        print(f"Processing batch file: {batch_file}")
        batch_data = load_jsonl(batch_file)
        normalized_data = normalize_json(batch_data)
        if not normalized_data.empty:
            all_data.append(normalized_data)

    # Concatenate all DataFrames into one
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No data found in batch files.")
        return pd.DataFrame()

# Load and parse the batches into a DataFrame
df = load_and_parse_batches(batch_files_dir)

# Function to safely parse JSON-like strings
def safe_json_loads(x):
    if isinstance(x, str):
        x = x.strip()  # Remove leading/trailing whitespace
        try:
            # Try parsing with json.loads first
            return json.loads(x)
        except (json.JSONDecodeError, TypeError):
            try:
                # Attempt to handle cases where single quotes are used instead of double quotes
                return json.loads(x.replace("'", "\""))
            except (json.JSONDecodeError, TypeError):
                try:
                    # If it's a literal like None, True, False
                    return ast.literal_eval(x)
                except (ValueError, SyntaxError):
                    # If all attempts fail, return the original string
                    return x
    else:
        return x

# Function to process columns containing lists and convert them to semicolon-separated strings
def process_list_columns(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(safe_json_loads)
            # If the cell is a list, join its elements into semicolon-separated string
            df[col] = df[col].apply(lambda x: '; '.join(map(str, x)) if isinstance(x, list) else x)
    return df

# List of array fields in the new response format
array_fields = [
    'authors', 'authors_institutions', 'authors_emails', 
    'all_institutions', 'jel_codes', 'keywords'
]

# Process array fields
df = process_list_columns(df, array_fields)

# Handle missing or improperly formatted entries by replacing NaNs with 'NA'
df.fillna('NA', inplace=True)

# Convert boolean fields from True/False to 'True'/'False' strings if desired
boolean_fields = [
    'is_finance', 'is_development', 'is_labour', 'is_public_economics',
    'is_urban_economics', 'is_macroeconomics', 'is_behavioral_economics',
    'is_economic_history', 'is_econometric_theory', 'is_industrial_organization',
    'is_environmental_economics', 'is_health_economics'
]
for field in boolean_fields:
    if field in df.columns:
        df[field] = df[field].apply(lambda x: 'True' if x else 'False')

# Convert integer fields to string if CSV consistency is desired
if 'year_of_release' in df.columns:
    df['year_of_release'] = df['year_of_release'].astype(str)

# Convert 'date_of_publication' to standardized format if possible
def format_date(date_str):
    try:
        return pd.to_datetime(date_str, dayfirst=True).strftime('%d/%m/%Y')
    except:
        return 'NA'

if 'date_of_publication' in df.columns:
    df['date_of_publication'] = df['date_of_publication'].apply(format_date)

# Save the flattened DataFrame to a CSV
output_csv = 'extracted_responses_new_format.csv'
df.to_csv(output_csv, index=False)

print(f"All JSON columns have been successfully flattened and saved to {output_csv}.")


Processing batch files:   2%|▏         | 1/50 [00:00<00:06,  7.05it/s]

Processing batch file: output_batches/batch_output_v7_batch_6706baed7d9881908c52c285f489f825.jsonl
Processing batch file: output_batches/batch_output_v7_batch_6706bb2225fc8190962a931136e18372.jsonl


Processing batch files:   6%|▌         | 3/50 [00:00<00:07,  6.39it/s]

Processing batch file: output_batches/batch_output_v7_batch_6706bb4e95dc819099d4628e3db38740.jsonl
Processing batch file: output_batches/batch_output_v7_batch_6706bb84a3e08190a03cf44ed965beb2.jsonl


Processing batch files:  10%|█         | 5/50 [00:00<00:06,  6.60it/s]

...

Processing batch file: output_batches/batch_output_v7_batch_6706c37116648190825727d07a20e51e.jsonl
Processing batch file: output_batches/batch_output_v7_batch_6706c39b1d1c8190b270f0f6adb5ab40.jsonl


Processing batch files:  98%|█████████▊| 49/50 [00:06<00:00,  7.38it/s]

Processing batch file: output_batches/batch_output_v7_batch_6706c3c4acac8190bfe6d7a2915b91b6.jsonl
Processing batch file: output_batches/batch_output_v7_batch_6706c3f149908190ac8d0cd255adf5ca.jsonl


Processing batch files: 100%|██████████| 50/50 [00:07<00:00,  7.11it/s]
<unknown>:1: SyntaxWarning: invalid decimal literal


All JSON columns have been successfully flattened and saved to extracted_responses_new_format.csv.
